# 其余67湖正式率定与评价：完整配置、12进程运行入口

按顺序运行全部Python代码单元，会在C盘生成新配置和完整R运行文件；notebook本身不会启动模型。
随后在原隔离项目RStudio运行最后显示的source命令，即自动检测12进程、只读核验已完成两湖，并正式执行其余67湖。

仍使用LakeEnsemblR 1.2.9.9003、四模型、每模型500套；Simstrat增加p_sw_ice=0.9–3，Puhu湖底70m同步修正。
旧两湖的10进程请求、结果与配置不修改，也不重新计算。新结果为D:/LER4_ice_20261103/n500_c12。


In [ ]:
PROJECT = "C:/Geography/R/dissertation_LakeEnsemblR"
ISO = ("D:/Geography/R/dissertation_LakeEnsemblR/LER_window/"
       "MyLake_v121_ordering_isolated_20261029")
LIBRARY = ISO + "/library_R4_6_mylake_v121_order9002"
LAKES_CSV = None  # 自动查找原69湖登记表；其他位置时填写完整路径。
DESTINATION = None  # 保持None：新建yaml/r4_ice_20261103_core12。


In [ ]:
"""Clone the current, repaired 69-lake YAMLs into an independent experiment.

Run in Windows Python or WSL. This file does not run R or any lake model.
"""
from pathlib import Path
import copy
import csv
import hashlib
import io
import json
import math
import os
import re
import yaml

MODELS = ("FLake", "GLM", "GOTM", "Simstrat")
SELECTED = ("akesaiqin", "namco")
VARIANT = "r4_ice_20261103_core12"
ICE_SPEC = {"lower": 0.9, "upper": 3.0, "initial": 1.0, "log": False}
MET_SPEC = {
    "wind_speed": {"lower": 0.5, "upper": 2.0, "initial": 1.0, "log": False},
    "swr": {"lower": 0.5, "upper": 1.5, "initial": 1.0, "log": False},
}
NATIVE = {
    "FLake": ("FLakeR", "flake.nml"), "GLM": ("GLM3r", "glm3.nml"),
    "GOTM": ("GOTMr", "gotm.yaml"), "Simstrat": ("SimstratR", "simstrat.par"),
}


class LERLoader(yaml.SafeLoader):
    pass


LERLoader.yaml_implicit_resolvers = {
    k: [(tag, rule) for tag, rule in rules if tag != "tag:yaml.org,2002:timestamp"]
    for k, rules in yaml.SafeLoader.yaml_implicit_resolvers.items()
}


def require(ok, message):
    if not ok:
        raise ValueError(message)


def host_path(value):
    """Interpret an existing Windows/WSL path on the current Python host."""
    s = str(value).replace("\\", "/")
    if os.name != "nt" and re.match(r"^[A-Za-z]:/", s):
        s = "/mnt/" + s[0].lower() + s[2:]
    elif os.name == "nt" and re.match(r"^/mnt/[A-Za-z]/", s):
        s = s[5].upper() + ":/" + s[7:]
    return Path(s).expanduser()


def windows_text(path):
    s = Path(path).as_posix()
    return s[5].upper() + ":/" + s[7:] if re.match(r"^/mnt/[A-Za-z]/", s) else s


def sha(data):
    return hashlib.sha256(data).hexdigest()


def csv_read(path):
    raw = Path(path).read_bytes()
    for enc in ("utf-8-sig", "gb18030"):
        try:
            return list(csv.DictReader(io.StringIO(raw.decode(enc))))
        except UnicodeError:
            continue
    raise ValueError(f"CSV编码无法识别: {path}")


def csv_bytes(rows):
    require(bool(rows), "CSV不能为空")
    fields = list(dict.fromkeys(k for row in rows for k in row))
    out = io.StringIO(newline="")
    writer = csv.DictWriter(out, fields, lineterminator="\n")
    writer.writeheader()
    writer.writerows(rows)
    return out.getvalue().encode("utf-8-sig")


def yaml_text(cfg):
    """Match the three-space layout required by the installed LER text reader."""
    def scalar(value):
        if isinstance(value, bool):
            return "true" if value else "false"
        if isinstance(value, (int, float)):
            require(math.isfinite(value), "YAML数值不是有限值")
            return re.sub(r"^(-?\d+)(e[+-]\d+)$", r"\1.0\2", format(value, ".17g"))
        require(isinstance(value, str), f"不支持的YAML值: {value!r}")
        if re.fullmatch(r"\d{4}-\d\d-\d\d \d\d:\d\d:\d\d", value):
            return value
        if re.fullmatch(r"[A-Za-z_][A-Za-z0-9_./-]*", value) and value.lower() not in {
            "null", "true", "false", "yes", "no", "on", "off"
        }:
            return value
        return json.dumps(value, ensure_ascii=False)

    lines = []
    def emit(mapping, depth=0):
        for key, value in mapping.items():
            require(bool(re.fullmatch(r"[A-Za-z_][A-Za-z0-9_./-]*", key)), f"非法YAML键: {key}")
            prefix = " " * depth + key + ":"
            if isinstance(value, dict):
                lines.append(prefix + ("" if value else " {}"))
                if value:
                    emit(value, depth + 3)
            elif isinstance(value, list):
                require(bool(value), f"空列表: {key}")
                lines.append(prefix)
                lines.extend(" " * (depth + 3) + "- " + scalar(v) for v in value)
            else:
                lines.append(prefix + " " + scalar(value))
    emit(cfg)
    text = "\n".join(lines) + "\n"
    require(yaml.load(text, Loader=LERLoader) == cfg, "YAML读回不一致")
    return text.encode("utf-8")


def simstrat_json(data):
    # Simstrat .par uses JSON with ! comments and occasionally trailing commas.
    text = data.decode("utf-8-sig")
    out, quoted, escaped, comment = [], False, False, False
    for c in text:
        if comment:
            if c == "\n":
                comment = False
                out.append(c)
            continue
        if not quoted and c == "!":
            comment = True
            continue
        out.append(c)
        if c == '"' and not escaped:
            quoted = not quoted
        escaped = c == "\\" and not escaped
    return json.loads(re.sub(r",\s*([}\]])", r"\1", "".join(out)))


def glm_evap_is_disabled(data):
    text = data.decode("utf-8-sig")
    # Remove Fortran comments outside quoted strings, then read the exact group.
    text = re.sub(r"(?m)!.*$", "", text)
    group = re.search(r"(?is)&debugging\b(.*?)(?:^\s*/|&end\b)", text, re.M)
    return bool(group and re.search(r"(?i)\bdisable_evap\s*=\s*(?:\.true\.|true\b|t\b)", group[1]))


def build_variant(project, registry, library, destination=None):
    project, registry, library = map(lambda p: host_path(p).resolve(), (project, registry, library))
    source = project / "yaml"
    target = host_path(destination).resolve() if destination else source / VARIANT
    require(source.is_dir() and target != source, "原YAML目录缺失或目标等于原目录")
    require(registry.is_file(), f"湖泊登记表不存在: {registry}")
    source_files = sorted(source.glob("*_LakeEnsemblR.yaml"))
    require(len(source_files) == 69, f"原目录应有69个YAML，实际{len(source_files)}")
    old_manifest = csv_read(source / "_yaml_manifest.csv")
    manifest_by_name = {r["name"]: r for r in old_manifest}
    registry_rows = csv_read(registry)
    registry_by_name = {r["name"].strip(): r for r in registry_rows}
    require(len(registry_by_name) == len(registry_rows) == 69, "登记表应为69个不重名湖泊")
    require(all(n in registry_by_name for n in SELECTED), "登记表缺少目标两湖")
    require(len(manifest_by_name) == len(old_manifest) == 69,
            "原YAML清单应为69个不重名湖泊")
    for name, row in manifest_by_name.items():
        try:
            spin = float(row["spin_up_days"])
        except (ValueError, TypeError, KeyError) as error:
            raise ValueError(f"无效的spin_up_days: {name}") from error
        require(math.isfinite(spin) and spin >= 0, f"spin_up_days必须为非负数: {name}")
    ids = []
    for name, row in registry_by_name.items():
        try:
            identifier = float(row.get("NAME_ID", ""))
        except (ValueError, TypeError) as error:
            raise ValueError(f"无效的NAME_ID: {name}") from error
        require(math.isfinite(identifier) and identifier > 0 and identifier == math.floor(identifier),
                f"NAME_ID必须为正整数: {name}")
        ids.append(int(identifier))
    require(len(set(ids)) == 69, "69湖NAME_ID不唯一")
    source_bytes = {str(p): p.read_bytes() for p in (registry, source / "_yaml_manifest.csv")}
    source_hashes = {p: sha(data) for p, data in source_bytes.items()}
    planned, designs, new_manifest, input_records = {}, [], [], []
    puhu_geometry = None
    previous_root = source / "r4_ice_20261102"
    previous_identity = {
        "previous_config_root": windows_text(previous_root),
        "checked": previous_root.is_dir(), "lakes": [],
        "comparison": "Master settings and actual input/native-template content SHA256; output filenames remain labels",
        "historical_snapshot_check": "The remaining-lakes R runner separately verifies the completed two sealed snapshots",
    }

    def remember(path):
        key = str(path)
        if key not in source_bytes:
            data = Path(path).read_bytes()
            source_bytes[key] = data
            source_hashes[key] = sha(data)
        return source_bytes[key]

    def plan(relative, data):
        relative = Path(relative).as_posix()
        require(relative not in planned or planned[relative] == data, f"文件冲突: {relative}")
        planned[relative] = data

    def rebase(value, label):
        if not isinstance(value, str) or value.upper() == "NULL" or value == "":
            return value
        p = host_path(value)
        p = (p if p.is_absolute() else source / p).resolve()
        require(p.is_file(), f"输入文件不存在 ({label}): {p}")
        drive = lambda path: re.match(r"^([A-Za-z]):/", windows_text(path))
        p_drive, target_drive = drive(p), drive(target)
        cross_drive = p_drive and target_drive and p_drive[1].lower() != target_drive[1].lower()
        if cross_drive:
            # Also needed under WSL: ../../d/... would be invalid in Windows R.
            data = remember(p)
            basename = re.sub(r"[^A-Za-z0-9_.-]", "_", p.name)
            relative = "_inputs/" + sha(data) + "_" + basename
            plan(relative, data)
        else:
            try:
                relative = os.path.relpath(p, target).replace("\\", "/")
            except ValueError as error:
                raise ValueError(f"无法生成输入相对路径 ({label}): {p}") from error
        require(not re.match(r"^[A-Za-z]:", relative), "禁止写入含盘符的YAML路径")
        input_records.append({"field": label, "path": windows_text(p), "bytes": p.stat().st_size,
                              "mtime_ns": p.stat().st_mtime_ns})
        return relative

    def rebase_files(node, label):
        if isinstance(node, dict):
            for k, v in node.items():
                node[k] = rebase(v, label + "/" + k) if k == "file" else rebase_files(v, label + "/" + k)
        elif isinstance(node, list):
            node = [rebase_files(v, label) for v in node]
        return node

    def content_sha(base, value):
        if not isinstance(value, str) or value.upper() == "NULL" or value == "":
            return value
        relative = Path(value.replace("\\", "/")).as_posix()
        if base == target and relative in planned:
            return sha(planned[relative])
        path = host_path(value)
        path = (path if path.is_absolute() else base / path).resolve()
        require(path.is_file(), f"已完成两湖配置对比缺少输入: {path}")
        key = str(path)
        if key not in source_hashes:
            digest = hashlib.sha256()
            with path.open("rb") as handle:
                for chunk in iter(lambda: handle.read(1024 * 1024), b""):
                    digest.update(chunk)
            source_hashes[key] = digest.hexdigest()
        return source_hashes[key]

    def canonical_config(cfg, base):
        result = copy.deepcopy(cfg)
        def fingerprint(value):
            digest = content_sha(base, value)
            return "sha256=" + digest if isinstance(digest, str) and len(digest) == 64 else digest
        def files(node):
            if isinstance(node, dict):
                for key, value in node.items():
                    node[key] = fingerprint(value) if key == "file" else files(value)
            elif isinstance(node, list):
                node = [files(value) for value in node]
            return node
        result["location"]["hypsograph"] = fingerprint(result["location"]["hypsograph"])
        for section in ("input", "observations", "inflows", "outflows"):
            if section in result:
                result[section] = files(result[section])
        kw = result.get("input", {}).get("light", {}).get("Kw")
        if isinstance(kw, dict):
            for key, value in kw.items():
                if isinstance(value, str) and not value.startswith("sha256="):
                    kw[key] = fingerprint(value)
        elif isinstance(kw, str):
            result["input"]["light"]["Kw"] = fingerprint(kw)
        for model, value in result["config_files"].items():
            result["config_files"][model] = fingerprint(value)
        return result

    for source_yaml in source_files:
        cfg = yaml.load(remember(source_yaml).decode("utf-8-sig"), Loader=LERLoader)
        name = cfg["location"]["name"]
        require(name in registry_by_name and name in manifest_by_name, f"登记表/清单缺湖: {name}")
        require(source_yaml.name == f"{name}_LakeEnsemblR.yaml", "YAML文件名与湖名不一致")
        require(str(cfg["time"]["start"]) == "1979-01-01 00:00:00" and
                str(cfg["time"]["stop"]) == "2016-01-01 00:00:00" and
                float(cfg["time"]["time_step"]) == 3600 and float(cfg["output"]["time_step"]) == 86400,
                f"模拟时段或时间步长与已完成两湖方案不一致: {name}")
        require(cfg["input"]["ice"]["use"] is True, f"原配置未开启湖冰: {name}")
        require(set(cfg["calibration"]).issubset({"met", *MODELS, "MyLake"}),
                f"发现本方案未定义的率定参数组，停止以免误删: {name}")
        require(cfg["calibration"]["met"] == MET_SPEC, f"原公共率定范围已变化: {name}")
        require(all(not cfg["calibration"].get(m, {}) for m in (*MODELS, "MyLake")),
                f"发现原生率定参数，需先核对，防止误删: {name}")
        cfg = copy.deepcopy(cfg)
        puhu_profile_relative = None
        if name == "puhu":
            # The user already removed the redundant 70.2 m / zero-area row.
            # Only the NEW master YAML is adjusted; never edit the original CSV.
            hyp_path = host_path(cfg["location"]["hypsograph"])
            hyp_path = (hyp_path if hyp_path.is_absolute() else source / hyp_path).resolve()
            require(hyp_path.is_file(), f"Puhu湖盆CSV不存在: {hyp_path}")
            remember(hyp_path)
            bathy = csv_read(hyp_path)
            require(bool(bathy) and {"Depth_meter", "Area_meterSquared"}.issubset(bathy[0]),
                    "Puhu湖盆CSV必须有Depth_meter/Area_meterSquared")
            depths = [float(r["Depth_meter"]) for r in bathy]
            areas = [float(r["Area_meterSquared"]) for r in bathy]
            require(all(math.isfinite(x) for x in depths + areas), "Puhu湖盆数据含缺失/无穷")
            require(depths[0] == 0 and abs(depths[-1] - 70.0) < 1e-8 and areas[-1] == 0,
                    "Puhu湖盆应从0m开始、在70m结束且底面积为0；不要自动删除或改写原CSV")
            require(all(a < b for a, b in zip(depths, depths[1:])) and
                    all(a > b >= 0 for a, b in zip(areas, areas[1:])),
                    "Puhu深度必须严格递增，面积必须严格递减至唯一零底面积")
            old_depth, old_init = cfg["location"]["depth"], cfg["location"]["init_depth"]
            require(abs(float(old_init) - float(old_depth)) < 1e-8,
                    "Puhu原init_depth与depth不一致，需先核对水位设置")
            cfg["location"]["depth"] = cfg["location"]["init_depth"] = 70.0
            glm = cfg["model_parameters"]["GLM"]
            removed = []
            for key in list(glm):
                if key in ("morphometry/H", "morphometry/A", "morphometry/bsn_vals"):
                    removed.append(key)
                    del glm[key]
                elif key.endswith("/lake_depth"):
                    glm[key] = 70.0
                elif key == "morphometry/crest_elev":
                    glm[key] = float(cfg["location"]["elevation"])
            puhu_geometry = {
                "name": name, "source_csv": windows_text(hyp_path),
                "source_csv_sha256": source_hashes[str(hyp_path)],
                "previous_depth_m": old_depth, "previous_init_depth_m": old_init,
                "depth_m": 70, "init_depth_m": 70, "bottom_area_m2": 0,
                "hypsograph_rows": len(depths), "removed_stale_GLM_overrides": removed,
                "native_geometry_source": "Official export_location regenerates H,A,bsn_vals from unchanged CSV",
                "expected_GLM_bottom_H": float(cfg["location"]["elevation"]) - 70,
                "expected_GLM_crest_H": float(cfg["location"]["elevation"]),
            }
            profile_path = host_path(cfg["input"]["init_temp_profile"]["file"])
            profile_path = (profile_path if profile_path.is_absolute() else source / profile_path).resolve()
            require(profile_path.is_file(), f"Puhu初温CSV不存在: {profile_path}")
            remember(profile_path)
            profile = csv_read(profile_path)
            require(bool(profile) and {"Depth_meter", "Water_Temperature_celsius"}.issubset(profile[0]),
                    "Puhu初温CSV必须有Depth_meter/Water_Temperature_celsius")
            profile_depth = [float(r["Depth_meter"]) for r in profile]
            profile_temp = [float(r["Water_Temperature_celsius"]) for r in profile]
            require(all(math.isfinite(x) for x in profile_depth + profile_temp) and profile_depth[0] == 0 and
                    all(a < b for a, b in zip(profile_depth, profile_depth[1:])),
                    "Puhu初温剖面必须从0m开始、深度严格递增且数值完整")
            kept = [r for r, d in zip(profile, profile_depth) if d <= 70.0]
            removed_depths = [d for d in profile_depth if d > 70.0]
            require(bool(kept) and float(kept[-1]["Depth_meter"]) == 70.0,
                    "Puhu初温必须已有70m层；不进行温度插值或外推")
            require(all(abs(d - 70.2) < 1e-8 for d in removed_depths),
                    "Puhu初温70m以下发现非70.2m层，需核对后再改")
            if removed_depths:
                profile_data = csv_bytes(kept)
                puhu_profile_relative = "_inputs/puhu_init_depth70_" + sha(profile_data) + ".csv"
                plan(puhu_profile_relative, profile_data)
            puhu_geometry.update(
                original_init_profile=windows_text(profile_path),
                original_init_profile_sha256=source_hashes[str(profile_path)],
                init_profile_removed_depths_m=removed_depths,
                init_profile_retained_rows=len(kept),
                init_profile_retained_values_unchanged=True,
                new_init_profile_relative=puhu_profile_relative,
            )
        cfg["config_files"] = {m: cfg["config_files"][m] for m in MODELS}
        for section in ("model_parameters", "scaling_factors"):
            cfg.get(section, {}).pop("MyLake", None)
        light_kw = cfg.get("input", {}).get("light", {}).get("Kw")
        if isinstance(light_kw, dict):
            light_kw.pop("MyLake", None)
            for k, value in light_kw.items():
                if isinstance(value, str) and value.upper() != "NULL":
                    light_kw[k] = rebase(value, name + "/input/light/Kw/" + k)
        elif isinstance(light_kw, str):
            cfg["input"]["light"]["Kw"] = rebase(light_kw, name + "/input/light/Kw")
        cfg["location"]["hypsograph"] = rebase(cfg["location"]["hypsograph"], name + "/hypsograph")
        for section in ("input", "observations", "inflows", "outflows"):
            if section in cfg:
                cfg[section] = rebase_files(cfg[section], name + "/" + section)
        if puhu_profile_relative:
            cfg["input"]["init_temp_profile"]["file"] = puhu_profile_relative
        cfg["calibration"] = {"met": copy.deepcopy(MET_SPEC), **{m: {} for m in MODELS}}
        cfg["calibration"]["Simstrat"]["ModelParameters/p_sw_ice"] = copy.deepcopy(ICE_SPEC)
        for model, relative in cfg["config_files"].items():
            rel = Path(str(relative).replace("\\", "/"))
            require(not rel.is_absolute() and ".." not in rel.parts and rel.parts[0] == model,
                    f"原生配置路径必须位于{model}/: {relative}")
            native = source / rel
            if not native.is_file():
                package, basename = NATIVE[model]
                require(model != "GLM", "必须保留已修复的yaml/GLM配置，禁止回退官方默认模板")
                require(rel.as_posix() == model + "/" + basename, f"定制模板缺失: {native}")
                native = library / package / "extdata" / basename
            require(native.is_file(), f"缺少原生模板: {native}")
            data = remember(native)
            if model == "GLM":
                require(glm_evap_is_disabled(data), f"GLM未确认debugging.disable_evap=TRUE: {native}")
            elif model == "Simstrat":
                native_cfg = simstrat_json(data)
                require(native_cfg["ModelConfig"]["IceModel"] == 1, "Simstrat模板IceModel必须为1")
                require(native_cfg["ModelConfig"]["Forcing"] == 5, "Simstrat模板Forcing必须为5")
                require(native_cfg["ModelParameters"]["p_sw_ice"] == 1, "原Simstrat p_sw_ice初值必须为1")
            plan(rel, data)
            for parameter, spec in MET_SPEC.items():
                designs.append({"name": name, "model": model, "category": "met", "parameter": parameter, **spec})
            if model == "Simstrat":
                designs.append({"name": name, "model": model, "category": "Simstrat",
                                "parameter": "ModelParameters/p_sw_ice", **ICE_SPEC})
        plan(source_yaml.name, yaml_text(cfg))
        if previous_root.is_dir() and name in SELECTED:
            previous_file = previous_root / source_yaml.name
            require(previous_file.is_file(), f"旧四模型配置缺少已完成湖: {previous_file}")
            previous_cfg = yaml.load(remember(previous_file).decode("utf-8-sig"), Loader=LERLoader)
            require(canonical_config(previous_cfg, previous_root) == canonical_config(cfg, target),
                    f"新配置与已完成两湖方案存在实际设置或输入内容差异: {name}；保留现状，不修改旧目录")
            previous_identity["lakes"].append({"name": name, "passed": True})
        row = dict(manifest_by_name[name])
        row.update(variant=VARIANT, native_parameters_enabled="True", parameter_count="9",
                   NAME_ID=registry_by_name[name]["NAME_ID"],
                   yaml_file=windows_text(target / source_yaml.name))
        row["depth_m"] = format(float(cfg["location"]["depth"]), ".17g")
        configured_paths = {
            "meteo_file": cfg["input"]["meteo"]["file"],
            "hypsograph_file": cfg["location"]["hypsograph"],
            "init_temp_file": cfg["input"]["init_temp_profile"]["file"],
        }
        for key, relative in configured_paths.items():
            row["original_" + key] = row.get(key, "")
            row[key] = windows_text((target / relative).resolve())
        new_manifest.append(row)
    require(len(designs) == 69 * 9, "参数设计应为69×9行")
    require({r["name"] for r in new_manifest} == set(registry_by_name), "69湖登记表不一致")
    all_rows = []
    manifest_new_by_name = {r["name"]: r for r in new_manifest}
    for original in registry_rows:
        name = original["name"].strip()
        row = dict(original)
        row["name"] = name
        row["spin_up_days"] = manifest_by_name[name]["spin_up_days"]
        row["configured_depth_m"] = manifest_new_by_name[name]["depth_m"]
        all_rows.append(row)
    all_by_name = {r["name"]: r for r in all_rows}
    selected_rows = [all_by_name[name] for name in SELECTED]
    remaining_rows = [r for r in all_rows if r["name"] not in SELECTED]
    require(len(remaining_rows) == 67 and len(all_rows) == 69,
            "运行清单应严格排除已完成两湖，剩余67湖")
    plan("all_lakes.csv", csv_bytes(all_rows))
    plan("remaining_lakes.csv", csv_bytes(remaining_rows))
    plan("selected_lakes.csv", csv_bytes(selected_rows))
    plan("_yaml_manifest.csv", csv_bytes(new_manifest))
    plan("parameter_design.csv", csv_bytes(designs))
    plan("selected_parameter_design.csv", csv_bytes([r for r in designs if r["name"] in SELECTED]))
    plan("remaining_parameter_design.csv", csv_bytes([r for r in designs if r["name"] not in SELECTED]))
    plan("_input_inventory.csv", csv_bytes(input_records))
    plan("_completed_two_config_identity.json",
         (json.dumps(previous_identity, ensure_ascii=False, indent=2) + "\n").encode("utf-8"))
    require(puhu_geometry is not None, "未找到Puhu湖盆审计")
    plan("_puhu_geometry_audit.json", (json.dumps(puhu_geometry, ensure_ascii=False, indent=2) + "\n").encode("utf-8"))
    metadata = {
        "variant": VARIANT, "yaml_root": windows_text(target), "original_yaml_root": windows_text(source),
        "models": MODELS, "selected_lakes": SELECTED, "generated_lakes": 69,
        "completed_lakes_excluded_from_remaining_run": SELECTED,
        "remaining_lakes": [r["name"] for r in remaining_rows], "remaining_lake_count": 67,
        "remaining_run_ncores": 12,
        "spin_up_days_preserved": True, "met_calibration": MET_SPEC,
        "Simstrat_calibration": {"ModelParameters/p_sw_ice": ICE_SPEC},
        "model_parameter_counts": {"FLake": 2, "GLM": 2, "GOTM": 2, "Simstrat": 3},
        "calibration_counts_configured_in_R": [500, 2000],
        "puhu_depth_m": 70, "puhu_original_hypsograph_preserved": True,
        "note": "User-specified p_sw_ice search domain; keep ice physics and other settings unchanged. No models started.",
    }
    plan("_yaml_settings.json", (json.dumps(metadata, ensure_ascii=False, indent=2) + "\n").encode("utf-8"))
    plan("_source_sha256.csv", csv_bytes([{"path": windows_text(p), "sha256": digest}
                                         for p, digest in sorted(source_hashes.items())]))
    plan("_generated_sha256.csv", csv_bytes([{"relative_path": p, "sha256": sha(data)}
                                            for p, data in sorted(planned.items())]))
    # Precheck every destination before creating or changing any file.
    for relative, data in planned.items():
        path = target / relative
        require(not path.exists() or (path.is_file() and path.read_bytes() == data),
                f"目标已有不同内容，停止以免覆盖: {path}")
    for path, digest in source_hashes.items():
        require(sha(Path(path).read_bytes()) == digest, f"生成期间原文件发生变化: {path}")
    for relative, data in planned.items():
        path = target / relative
        path.parent.mkdir(parents=True, exist_ok=True)
        if not path.exists():
            with path.open("xb") as handle:
                handle.write(data)
    print("REMAINING67_CONFIG_READY_NO_MODELS_STARTED")
    print("YAML:", windows_text(target))
    print("Generated lakes: 69; remaining: 67; excluded completed: akesaiqin, namco")
    print("Design rows: all 621; remaining 603; completed two 18")
    return metadata


In [ ]:
project = host_path(PROJECT)
if LAKES_CSV is None:
    candidates = [
        project / "lake_joinWan_pts_70L.csv",
        host_path("C:/Geography/GLM_model/simstratdata/sim1_input/lake_joinWan_pts_70L.csv"),
    ]
    existing = [p for p in candidates if p.is_file()]
    require(bool(existing), "未找到原69湖登记表，请填写第一个代码单元的LAKES_CSV")
    if len(existing) > 1:
        identities = [{(r["name"].strip(), str(r["NAME_ID"])) for r in csv_read(p)}
                      for p in existing]
        require(all(x == identities[0] for x in identities),
                "两个候选登记表的NAME_ID/name不同，请明确填写LAKES_CSV")
    lake_csv = existing[0]
else:
    lake_csv = host_path(LAKES_CSV)
CONFIG = build_variant(PROJECT, lake_csv, LIBRARY, DESTINATION)


执行完成必须看到`REMAINING67_CONFIG_READY_NO_MODELS_STARTED`。

默认输出：`C:/Geography/R/dissertation_LakeEnsemblR/yaml/r4_ice_20261103_core12`。

旧的`yaml/r4_ice_20261102`及`D:/LER4_ice_20261102/n500`均不修改。若旧四模型配置目录存在，代码会对已完成两湖的主设置、输入文件内容和原生模板做同一性核对；R运行入口还会核对已封存结果中的原始快照。

如果目标文件存在且内容一致，可重复运行；若内容不同则停止并保留文件。不要通过删除旧回执或关闭校验来继续。

配置检查和生成在这里结束。正式模拟使用单独的剩余67湖R入口，该入口只运行remaining_lakes.csv中的67湖，使用12进程、每模型500套。


## 保存完整运行器

本单元已经嵌入全部R脚本和恢复helper，运行后不需要额外寻找或下载脚本。

In [ ]:
# 本单元只保存完整脚本，不启动模型。
import hashlib
import json
config_root = host_path(CONFIG["yaml_root"])
def save_identical_or_new(path, content):
    data = content.encode("utf-8")
    require(not path.exists() or path.read_bytes() == data,
            f"已有不同脚本，停止以免覆盖研究来源: {path}")
    if not path.exists():
        with path.open("xb") as handle:
            handle.write(data)
    return hashlib.sha256(data).hexdigest()

SCRIPT_CONTENTS = {'Resume_interrupted_calibration_9003_rev3.R': '# Resume one interrupted LakeEnsemblR 1.2.9.9003 window calibration without\n# changing the installed package, parameter pool, model equations, or objective.\n#\n# The original public API writes parameters.csv before a candidate starts and\n# status.csv near the end.  This script validates every retained candidate,\n# preserves incomplete candidate directories outside the sealed calibration\n# run, executes only missing candidates, and then reproduces the original\n# cali_window_ensemble() finalization and R69 calibration receipt.\n\n.rr_resume_source_file <- local({\n  path <- tryCatch(sys.frame(1L)$ofile, error = function(e) NULL)\n  if (is.null(path) || length(path) != 1L || !nzchar(path)) return(NA_character_)\n  normalizePath(path, winslash = "/", mustWork = TRUE)\n})\n\n.rr_need <- function(ok, ...) {\n  if (!isTRUE(ok)) stop(..., call. = FALSE)\n}\n\n.rr_fun <- function(R69, name) {\n  fun <- tryCatch(R69[[name]], error = function(e) NULL)\n  .rr_need(is.function(fun), "R69 does not provide function: ", name)\n  fun\n}\n\n.rr_norm <- function(path, mustWork = TRUE) {\n  normalizePath(path, winslash = "/", mustWork = mustWork)\n}\n\n.rr_stamp <- function() {\n  format(Sys.time(), "%Y%m%dT%H%M%SZ", tz = "UTC")\n}\n\n.rr_sha <- function(path) {\n  digest::digest(file = path, algo = "sha256", serialize = FALSE)\n}\n\n.rr_read_csv <- function(path) {\n  .rr_need(file.exists(path) && !dir.exists(path), "Missing CSV: ", path)\n  out <- utils::read.csv(path, stringsAsFactors = FALSE, check.names = FALSE,\n                         fileEncoding = "UTF-8-BOM")\n  .rr_need(!anyDuplicated(names(out)), "Duplicate CSV columns: ", path)\n  out\n}\n\n.rr_numeric_equal <- function(a, b, tolerance = 1e-12) {\n  a <- suppressWarnings(as.numeric(a))\n  b <- suppressWarnings(as.numeric(b))\n  if (length(a) != length(b)) return(FALSE)\n  both_na <- is.na(a) & is.na(b)\n  if (any(xor(is.na(a), is.na(b)))) return(FALSE)\n  take <- !both_na\n  if (!any(take)) return(TRUE)\n  if (any(is.infinite(a[take]) | is.infinite(b[take]))) {\n    same_inf <- is.infinite(a[take]) & is.infinite(b[take]) &\n      sign(a[take]) == sign(b[take])\n    finite <- is.finite(a[take]) & is.finite(b[take])\n    if (any(!(same_inf | finite))) return(FALSE)\n    if (!any(finite)) return(TRUE)\n    aa <- a[take][finite]\n    bb <- b[take][finite]\n  } else {\n    aa <- a[take]\n    bb <- b[take]\n  }\n  all(abs(aa - bb) <= tolerance * pmax(1, abs(aa), abs(bb)))\n}\n\n.rr_table_equal <- function(actual, expected, label, tolerance = 1e-12) {\n  .rr_need(is.data.frame(actual) && is.data.frame(expected),\n           label, ": expected data frames")\n  .rr_need(identical(names(actual), names(expected)),\n           label, ": column names/order differ")\n  .rr_need(nrow(actual) == nrow(expected), label, ": row count differs")\n  for (name in names(expected)) {\n    want <- expected[[name]]\n    got <- actual[[name]]\n    if (inherits(want, "POSIXt")) {\n      want <- format(want, "%Y-%m-%d %H:%M:%S", tz = "UTC")\n    }\n    if (is.numeric(want) || is.integer(want)) {\n      .rr_need(.rr_numeric_equal(got, want, tolerance),\n               label, ": numeric field differs: ", name)\n    } else if (is.logical(want)) {\n      normalize_bool <- function(x) {\n        x <- toupper(as.character(x))\n        x[x == "1"] <- "TRUE"\n        x[x == "0"] <- "FALSE"\n        x\n      }\n      .rr_need(identical(normalize_bool(got), normalize_bool(want)),\n               label, ": logical field differs: ", name)\n    } else {\n      .rr_need(identical(as.character(got), as.character(want)),\n               label, ": text field differs: ", name)\n    }\n  }\n  invisible(TRUE)\n}\n\n.rr_candidate_required <- function(ctx, status) {\n  files <- c("parameters.csv", "contract.csv", "status.csv", "warnings.txt")\n  if (identical(status, "ok")) {\n    files <- c(files, "native_audit.csv", "pairs.csv", "metrics.csv")\n    if (isTRUE(ctx$control$save_series)) files <- c(files, "native_surface.rds")\n  }\n  files\n}\n\n.rr_validate_candidate <- function(ctx, model, pool_row, directory) {\n  id <- as.character(pool_row$par_id)\n  parameter_file <- file.path(directory, "parameters.csv")\n  status_file <- file.path(directory, "status.csv")\n  if (!dir.exists(directory)) {\n    return(list(state = "missing", status = NULL, reason = "directory absent"))\n  }\n  if (!file.exists(status_file)) {\n    return(list(state = "incomplete", status = NULL, reason = "status.csv absent"))\n  }\n  .rr_need(file.exists(parameter_file),\n           "Candidate has status.csv but no parameters.csv: ", directory)\n  parameters <- .rr_read_csv(parameter_file)\n  .rr_table_equal(parameters, pool_row, paste(model, id, "parameters"))\n\n  status <- .rr_read_csv(status_file)\n  .rr_need(nrow(status) == 1L &&\n             all(c("model", "par_id", "random_seed", "status", "stage",\n                   "message", "N", "RMSE", "MAE", "bias", "NSE", "r",\n                   "objective") %in% names(status)),\n           "Invalid status schema: ", status_file)\n  .rr_need(identical(as.character(status$model), model) &&\n             identical(as.character(status$par_id), id),\n           "Candidate status model/par_id mismatch: ", directory)\n  expected_seed <- get(".lw_seed", asNamespace("LakeEnsemblR"))(\n    ctx$control$seed, model, id)\n  .rr_need(.rr_numeric_equal(status$random_seed, expected_seed, 0),\n           "Candidate seed mismatch: ", directory)\n  state <- as.character(status$status)\n  .rr_need(length(state) == 1L && state %in% c("ok", "failed"),\n           "Invalid candidate status: ", directory)\n  required <- .rr_candidate_required(ctx, state)\n  absent <- required[!file.exists(file.path(directory, required))]\n  if (length(absent)) {\n    return(list(state = "incomplete", status = status,\n                reason = paste("missing", paste(absent, collapse = ", "))))\n  }\n\n  contract <- .rr_read_csv(file.path(directory, "contract.csv"))\n  expected_contract <- ctx$contracts[ctx$contracts$model == model, , drop = FALSE]\n  .rr_table_equal(contract, expected_contract,\n                  paste(model, id, "contract"))\n\n  if (identical(state, "ok")) {\n    audit <- .rr_read_csv(file.path(directory, "native_audit.csv"))\n    .rr_need(nrow(audit) == 1L && "model" %in% names(audit) &&\n               identical(as.character(audit$model), model),\n             "Invalid native_audit.csv: ", directory)\n    pairs <- .rr_read_csv(file.path(directory, "pairs.csv"))\n    .rr_need(all(c("window_id", "observed", "predicted") %in% names(pairs)) &&\n               nrow(pairs) == nrow(ctx$cal) && !anyDuplicated(pairs$window_id) &&\n               setequal(as.character(pairs$window_id), as.character(ctx$cal$window_id)),\n             "Invalid calibration pairs: ", directory)\n    ordered_pairs <- pairs[match(as.character(ctx$cal$window_id),\n                                 as.character(pairs$window_id)), , drop = FALSE]\n    .rr_need(.rr_numeric_equal(ordered_pairs$observed, ctx$cal$observed, 1e-12) &&\n               all(is.finite(suppressWarnings(as.numeric(ordered_pairs$predicted)))),\n             "Calibration observations/predictions are invalid: ", directory)\n    metrics <- .rr_read_csv(file.path(directory, "metrics.csv"))\n    .rr_need(nrow(metrics) == 1L &&\n               all(c("N", "RMSE", "MAE", "bias", "NSE", "r") %in% names(metrics)),\n             "Invalid metrics.csv: ", directory)\n    .rr_need(.rr_numeric_equal(metrics$N, nrow(ctx$cal), 0) &&\n               is.finite(suppressWarnings(as.numeric(metrics$RMSE))) &&\n               suppressWarnings(as.numeric(metrics$RMSE)) >= 0,\n             "Successful candidate metrics are not finite/complete: ", directory)\n    for (field in c("N", "RMSE", "MAE", "bias", "NSE", "r")) {\n      .rr_need(.rr_numeric_equal(status[[field]], metrics[[field]], 1e-12),\n               "status/metrics mismatch for ", model, "/", id, ": ", field)\n    }\n    .rr_need(.rr_numeric_equal(status$objective, metrics$RMSE, 1e-12),\n             "objective/RMSE mismatch for ", model, "/", id)\n  } else {\n    .rr_need(is.infinite(suppressWarnings(as.numeric(status$objective))) &&\n               suppressWarnings(as.numeric(status$objective)) > 0,\n             "Failed candidate objective must be Inf: ", directory)\n  }\n  list(state = "complete", status = status, reason = "")\n}\n\n.rr_scan_candidates <- function(ctx) {\n  rows <- list()\n  statuses <- list()\n  for (model in ctx$models) {\n    pool <- ctx$pools[[model]]\n    .rr_need(is.data.frame(pool) && "par_id" %in% names(pool) &&\n               !anyDuplicated(pool$par_id), "Invalid context pool: ", model)\n    model_root <- file.path(ctx$result_dir, "calibration", model)\n    if (dir.exists(model_root)) {\n      existing <- list.dirs(model_root, full.names = FALSE, recursive = FALSE)\n      extra <- setdiff(existing, as.character(pool$par_id))\n      .rr_need(!length(extra), "Unexpected candidate directories in ", model_root,\n               ": ", paste(extra, collapse = ", "))\n    }\n    for (i in seq_len(nrow(pool))) {\n      row <- pool[i, , drop = FALSE]\n      id <- as.character(row$par_id)\n      directory <- file.path(model_root, id)\n      checked <- .rr_validate_candidate(ctx, model, row, directory)\n      rows[[length(rows) + 1L]] <- data.frame(\n        model = model, par_id = id, state = checked$state,\n        reason = checked$reason, directory = directory,\n        directory_exists = dir.exists(directory),\n        parameters_exists = file.exists(file.path(directory, "parameters.csv")),\n        status_exists = file.exists(file.path(directory, "status.csv")),\n        stringsAsFactors = FALSE)\n      if (identical(checked$state, "complete")) {\n        statuses[[paste(model, id, sep = "/")]] <- checked$status\n      }\n    }\n  }\n  list(index = do.call(rbind, rows), statuses = statuses)\n}\n\n.rr_registry_row <- function(CFG, R69, lake) {\n  .rr_fun(R69, "r69_load_runtime")(CFG)\n  registry <- .rr_fun(R69, "r69_read_index")(CFG$lake_index, CFG$expected_lakes)\n  registry$spin_up_days <- CFG$spin_up_days\n  manifest <- file.path(CFG$yaml_dir, "_yaml_manifest.csv")\n  if (isTRUE(CFG$use_yaml_spinup_manifest) && file.exists(manifest)) {\n    values <- utils::read.csv(manifest, stringsAsFactors = FALSE,\n                              fileEncoding = "UTF-8-BOM")\n    .rr_need(all(c("name", "spin_up_days") %in% names(values)) &&\n               !anyDuplicated(values$name), "Invalid _yaml_manifest.csv")\n    positions <- match(registry$name, values$name)\n    .rr_need(!anyNA(positions), "YAML manifest does not contain all selected lakes")\n    spin <- suppressWarnings(as.numeric(values$spin_up_days[positions]))\n    .rr_need(all(is.finite(spin) & spin >= 0), "Invalid YAML spin-up days")\n    registry$spin_up_days <- spin\n  }\n  row <- registry[registry$name == lake, , drop = FALSE]\n  .rr_need(nrow(row) == 1L, "Lake is not a unique registry row: ", lake)\n  row\n}\n\n.rr_verify_live_request <- function(CFG, R69, lake, request, lake_root) {\n  live <- .rr_fun(R69, "r69_preflight")(\n    CFG, .rr_registry_row(CFG, R69, lake))\n  .rr_need(isTRUE(live$passed), "Live R69 preflight failed: ",\n           paste(live$issues, collapse = " | "))\n  .rr_need(identical(live$signature, request$signature),\n           "Live R69 input/runtime signature differs from the interrupted request")\n  acceptance <- .rr_fun(R69, "r69_read_sealed")(\n    file.path(lake_root, "acceptance", "acceptance_receipt.rds"))\n  .rr_need(identical(acceptance, request$acceptance),\n           "The sealed acceptance receipt differs from request$acceptance")\n  .rr_need(identical(acceptance$signature, request$signature) &&\n             isTRUE(acceptance$passed) && isTRUE(acceptance$parallel_accepted),\n           "Acceptance is not the required passed parallel acceptance")\n  live\n}\n\n.rr_run_lb_tasks <- function(tasks, ctx, ncores, log_file) {\n  if (!length(tasks)) return(list())\n  paths <- .libPaths()\n  cluster <- parallel::makePSOCKcluster(min(ncores, length(tasks)), outfile = log_file)\n  context_key <- paste0(\n    ".rr_worker_context_",\n    substr(digest::digest(normalizePath(log_file, winslash = "/", mustWork = FALSE),\n                          algo = "sha256", serialize = FALSE), 1L, 20L))\n  cleanup <- function() {\n    if (is.null(cluster)) return(invisible(NULL))\n    try(parallel::clusterCall(cluster, function(key) {\n      if (exists(key, envir = .GlobalEnv, inherits = FALSE))\n        rm(list = key, envir = .GlobalEnv)\n      TRUE\n    }, context_key), silent = TRUE)\n    try(parallel::stopCluster(cluster), silent = TRUE)\n    invisible(NULL)\n  }\n  on.exit(cleanup(), add = TRUE)\n  checks <- parallel::clusterCall(cluster, function(paths, ctx, key) {\n    .libPaths(paths)\n    get(".lw_initialize_runtime", asNamespace("LakeEnsemblR"))(paths)\n    get(".lw_verify_runtime", asNamespace("LakeEnsemblR"))(ctx)\n    assign(key, ctx, envir = .GlobalEnv)\n    TRUE\n  }, paths, ctx, context_key)\n  .rr_need(all(vapply(checks, isTRUE, logical(1))), "PSOCK worker verification failed")\n  results <- parallel::parLapplyLB(\n    cluster, tasks,\n    function(task, key) {\n      worker_ctx <- get(key, envir = .GlobalEnv, inherits = FALSE)\n      get(".lw_worker", asNamespace("LakeEnsemblR"))(task, worker_ctx)\n    },\n    key = context_key, chunk.size = 1L)\n  cleanup()\n  cluster <- NULL\n  results\n}\n\n.rr_concurrency_precheck <- function(ctx, scan, ncores, evidence_dir,\n                                     tolerance) {\n  .rr_need(ncores >= 10L, "The accepted 10-concurrent replay requires ncores >= 10")\n  replay_root <- file.path(evidence_dir, "five_model_10_concurrent_replay")\n  .rr_need(dir.create(replay_root, recursive = TRUE, showWarnings = FALSE),\n           "Could not create five-model replay directory")\n  replay_ctx <- ctx\n  replay_ctx$result_dir <- .rr_norm(replay_root)\n  rows <- list()\n  for (model in ctx$models) {\n    ids <- sort(scan$index$par_id[\n      scan$index$model == model & scan$index$state == "complete"])\n    ok <- vapply(ids, function(id) {\n      status <- scan$statuses[[paste(model, id, sep = "/")]]\n      !is.null(status) && identical(as.character(status$status), "ok")\n    }, logical(1))\n    ids <- ids[ok]\n    .rr_need(length(ids) >= 10L,\n             "Need at least 10 retained ok candidates for replay: ", model)\n    ids <- ids[seq_len(10L)]\n    pool <- ctx$pools[[model]]\n    positions <- match(ids, as.character(pool$par_id))\n    .rr_need(!anyNA(positions), "Replay IDs are absent from pool: ", model)\n    tasks <- lapply(positions, function(position) list(\n      model = model, spec = ctx$specs[[model]],\n      pool = pool[position, , drop = FALSE],\n      worker_id = sprintf("pc%04d", position)))\n    .rr_run_lb_tasks(tasks, replay_ctx, 10L,\n                     file.path(evidence_dir, paste0(model, "_precheck_workers.log")))\n    for (id in ids) {\n      original_dir <- file.path(ctx$result_dir, "calibration", model, id)\n      replay_dir <- file.path(replay_root, "calibration", model, id)\n      original_status <- .rr_read_csv(file.path(original_dir, "status.csv"))\n      replay_status <- .rr_read_csv(file.path(replay_dir, "status.csv"))\n      .rr_need(identical(names(original_status), names(replay_status)) &&\n                 all(vapply(c("model", "par_id", "status", "stage", "message"),\n                            function(field) identical(as.character(original_status[[field]]),\n                                                      as.character(replay_status[[field]])),\n                            logical(1))) &&\n                 identical(as.character(original_status$status), "ok"),\n               "Concurrency replay did not reproduce ok: ", model, "/", id)\n      for (field in c("random_seed", "N", "RMSE", "MAE", "bias", "NSE", "r",\n                      "objective")) {\n        .rr_need(.rr_numeric_equal(original_status[[field]], replay_status[[field]],\n                                   tolerance),\n                 "Replay status differs for ", model, "/", id, "/", field)\n      }\n      original_pairs <- .rr_read_csv(file.path(original_dir, "pairs.csv"))\n      replay_pairs <- .rr_read_csv(file.path(replay_dir, "pairs.csv"))\n      .rr_need(identical(as.character(original_pairs$window_id),\n                         as.character(replay_pairs$window_id)) &&\n                 .rr_numeric_equal(original_pairs$observed, replay_pairs$observed, 1e-12) &&\n                 .rr_numeric_equal(original_pairs$predicted, replay_pairs$predicted, tolerance),\n               "Replay window pairs differ: ", model, "/", id)\n      delta <- max(abs(as.numeric(original_pairs$predicted) -\n                         as.numeric(replay_pairs$predicted)), na.rm = TRUE)\n      rows[[length(rows) + 1L]] <- data.frame(\n        model = model, par_id = id, status_match = TRUE,\n        window_pairs = nrow(original_pairs), max_prediction_difference = delta,\n        tolerance = tolerance, passed = TRUE, stringsAsFactors = FALSE)\n    }\n  }\n  report <- do.call(rbind, rows)\n  .rr_write_evidence_csv(report,\n                         file.path(evidence_dir, "five_model_10_concurrent_replay.csv"))\n  report\n}\n\n.rr_validate_pool_files <- function(ctx) {\n  for (model in ctx$models) {\n    .rr_need(!is.null(ctx$specs[[model]]) && !is.null(ctx$pools[[model]]),\n             "Context lacks spec/pool: ", model)\n    pool_file <- file.path(ctx$result_dir, paste0("params_", model, ".csv"))\n    spec_file <- file.path(ctx$result_dir, paste0("spec_", model, ".csv"))\n    .rr_table_equal(.rr_read_csv(pool_file), ctx$pools[[model]],\n                    paste(model, "parameter pool"))\n    .rr_table_equal(.rr_read_csv(spec_file), ctx$specs[[model]],\n                    paste(model, "parameter specification"))\n  }\n  invisible(TRUE)\n}\n\n.rr_write_evidence_csv <- function(x, path) {\n  .rr_need(!file.exists(path), "Evidence file already exists: ", path)\n  utils::write.csv(x, path, row.names = FALSE, na = "NA", fileEncoding = "UTF-8")\n  invisible(path)\n}\n\n.rr_retained_manifest <- function(ctx, scan) {\n  manifest_fun <- get(".lw_file_manifest", asNamespace("LakeEnsemblR"))\n  rows <- list()\n  retained <- scan$index[scan$index$state == "complete", , drop = FALSE]\n  for (i in seq_len(nrow(retained))) {\n    manifest <- manifest_fun(retained$directory[i])\n    manifest$model <- retained$model[i]\n    manifest$par_id <- retained$par_id[i]\n    rows[[length(rows) + 1L]] <- manifest[\n      c("model", "par_id", "file", "bytes", "sha256")]\n  }\n  if (!length(rows)) {\n    return(data.frame(model = character(), par_id = character(),\n                      file = character(), bytes = numeric(), sha256 = character()))\n  }\n  out <- do.call(rbind, rows)\n  out <- out[order(match(out$model, ctx$models), out$par_id, out$file), , drop = FALSE]\n  rownames(out) <- NULL\n  out\n}\n\n.rr_verify_locked_manifest <- function(ctx, locked) {\n  .rr_need(is.data.frame(locked) &&\n             all(c("model", "par_id", "file", "bytes", "sha256") %in% names(locked)) &&\n             nrow(locked) > 0L, "Invalid locked retained-candidate manifest")\n  for (i in seq_len(nrow(locked))) {\n    path <- file.path(ctx$result_dir, "calibration", locked$model[i],\n                      locked$par_id[i], locked$file[i])\n    .rr_need(file.exists(path) && !dir.exists(path),\n             "A retained candidate artifact disappeared: ", path)\n    .rr_need(identical(as.numeric(file.info(path)$size), as.numeric(locked$bytes[i])) &&\n               identical(.rr_sha(path), as.character(locked$sha256[i])),\n             "A retained candidate artifact changed: ", path)\n  }\n  invisible(TRUE)\n}\n\n.rr_valid_prior_evidence <- function(attempt) {\n  normalized_attempt <- .rr_norm(attempt)\n  directories <- list.dirs(attempt, full.names = TRUE, recursive = FALSE)\n  directories <- directories[grepl("^resume_evidence_", basename(directories))]\n  directories[vapply(directories, function(directory) {\n    identity_file <- file.path(directory, "recovery_identity.txt")\n    manifest_file <- file.path(directory, "locked_retained_manifest.csv")\n    if (!file.exists(identity_file) || !file.exists(manifest_file)) return(FALSE)\n    identity <- readLines(identity_file, warn = FALSE)\n    identical(identity[startsWith(identity, "attempt=")],\n              paste0("attempt=", normalized_attempt)) &&\n      any(identical(identity[startsWith(identity, "LakeEnsemblR=")],\n                    "LakeEnsemblR=1.2.9.9003"))\n  }, logical(1))]\n}\n\n.rr_new_evidence_dir <- function(attempt) {\n  base <- paste0("resume_evidence_", .rr_stamp())\n  directory <- file.path(attempt, base)\n  suffix <- 0L\n  while (file.exists(directory)) {\n    suffix <- suffix + 1L\n    directory <- file.path(attempt, paste0(base, "_", suffix))\n  }\n  .rr_need(dir.create(directory, recursive = TRUE, showWarnings = FALSE),\n           "Could not create recovery evidence directory")\n  directory\n}\n\n.rr_write_or_verify_package_csv <- function(x, target, evidence_dir) {\n  writer <- get(".lw_write_csv", asNamespace("LakeEnsemblR"))\n  expected <- file.path(evidence_dir, paste0("expected_", basename(target)))\n  writer(x, expected)\n  if (file.exists(target)) {\n    .rr_need(identical(.rr_sha(target), .rr_sha(expected)),\n             "Existing finalization file differs from expected: ", target)\n  } else {\n    .rr_need(file.copy(expected, target, overwrite = FALSE),\n             "Could not install finalization file: ", target)\n  }\n  invisible(target)\n}\n\n.rr_copy_or_verify <- function(source, target) {\n  .rr_need(file.exists(source) && !dir.exists(source), "Missing source file: ", source)\n  dir.create(dirname(target), recursive = TRUE, showWarnings = FALSE)\n  if (file.exists(target)) {\n    .rr_need(identical(.rr_sha(source), .rr_sha(target)),\n             "Existing provenance file differs: ", target)\n  } else {\n    .rr_need(file.copy(source, target, overwrite = FALSE),\n             "Could not copy provenance file: ", target)\n  }\n  invisible(target)\n}\n\n.rr_atomic_context_replace <- function(context_file, ctx, evidence_dir) {\n  before <- file.path(evidence_dir, "context_before_finalize.rds")\n  ready <- file.path(evidence_dir, "context_ready.rds")\n  after <- file.path(evidence_dir, "context_after_finalize.rds")\n  .rr_need(!file.exists(before) && !file.exists(ready) && !file.exists(after),\n           "Context finalization evidence paths already exist")\n  saveRDS(ctx, ready, version = 3L)\n  .rr_need(file.rename(context_file, before),\n           "Could not atomically preserve the initial context.rds")\n  installed <- file.rename(ready, context_file)\n  if (!isTRUE(installed)) {\n    rollback <- file.rename(before, context_file)\n    .rr_need(isTRUE(rollback),\n             "New context install and rollback both failed; preserve the evidence directory")\n    stop("Could not atomically install finalized context.rds; original restored", call. = FALSE)\n  }\n  .rr_need(file.copy(context_file, after, overwrite = FALSE),\n           "Finalized context installed but its evidence copy failed")\n  invisible(context_file)\n}\n\n.rr_complete_receipt <- function(CFG, R69, attempt, ctx, request, summary,\n                                 evidence_dir, script_sha256) {\n  minimum <- if (is.null(CFG$min_success_fraction)) 0.95 else\n    suppressWarnings(as.numeric(CFG$min_success_fraction))\n  .rr_need(length(minimum) == 1L && is.finite(minimum) &&\n             minimum > 0 && minimum <= 1,\n           "Invalid CFG$min_success_fraction")\n  for (model in ctx$models) {\n    part <- summary[summary$model == model, , drop = FALSE]\n    .rr_need(nrow(part) == as.integer(request$num),\n             "Incomplete candidate count after recovery: ", model)\n    fraction <- mean(part$status == "ok" & is.finite(part$RMSE))\n    .rr_need(fraction >= minimum, model, ": successful candidate fraction ",\n             fraction, " is below ", minimum)\n  }\n\n  selected <- list(\n    signature = request$signature,\n    run_dir = .rr_norm(ctx$result_dir),\n    attempt_dir = .rr_norm(attempt),\n    success_count = sum(summary$status == "ok"),\n    failure_count = sum(summary$status != "ok"),\n    num = as.integer(request$num),\n    seed = request$seed\n  )\n  receipt_file <- file.path(attempt, "calibration_receipt.rds")\n  read_sealed <- .rr_fun(R69, "r69_read_sealed")\n  save_sealed <- .rr_fun(R69, "r69_save_sealed")\n  if (file.exists(receipt_file) || file.exists(paste0(receipt_file, ".sha256"))) {\n    old <- read_sealed(receipt_file)\n    .rr_need(identical(old, selected),\n             "Existing calibration receipt differs; it was not replaced")\n  } else {\n    save_sealed(selected, receipt_file)\n  }\n\n  recovery <- list(\n    lake = request$name,\n    attempt_dir = selected$attempt_dir,\n    run_dir = selected$run_dir,\n    retained_candidates = nrow(summary),\n    success_count = selected$success_count,\n    failure_count = selected$failure_count,\n    script_sha256 = script_sha256,\n    completed_UTC = format(Sys.time(), "%Y-%m-%d %H:%M:%S", tz = "UTC")\n  )\n  save_sealed(recovery, file.path(evidence_dir, "recovery_completion.rds"))\n  selected\n}\n\nresume_interrupted_lake_calibration <- function(\n    CFG,\n    R69,\n    lake = "akesaiqin",\n    ncores = 10L,\n    expected_completed = c(FLake = 334L, GLM = 430L, GOTM = 240L,\n                           Simstrat = 80L, MyLake = 17L),\n    expected_started = c(FLake = 336L, GLM = 432L, GOTM = 242L,\n                         Simstrat = 82L, MyLake = 19L),\n    attempt_dir = NULL,\n    execute_missing = TRUE,\n    worker_runner = NULL,\n    precheck_runner = NULL) {\n  .rr_need(requireNamespace("digest", quietly = TRUE), "digest is required")\n  .rr_need(identical(as.character(utils::packageVersion("LakeEnsemblR")),\n                     "1.2.9.9003"),\n           "This recovery is locked to LakeEnsemblR 1.2.9.9003; found ",\n           as.character(utils::packageVersion("LakeEnsemblR")))\n  .rr_need(is.character(lake) && length(lake) == 1L &&\n             grepl("^[A-Za-z0-9][A-Za-z0-9_-]*$", lake), "Invalid lake name")\n  ncores <- suppressWarnings(as.integer(ncores))\n  .rr_need(length(ncores) == 1L && is.finite(ncores) && ncores >= 1L,\n           "ncores must be a positive integer")\n  .rr_need(!is.null(CFG$results_root) && dir.exists(CFG$results_root),\n           "CFG$results_root does not exist")\n  .rr_need(isTRUE(CFG$resume), "CFG$resume must remain TRUE")\n  .rr_need(as.integer(CFG$num) == 500L, "CFG$num must remain 500")\n  .rr_need(as.integer(CFG$ncores) == ncores,\n           "Use the accepted CFG$ncores value; expected ", CFG$ncores)\n  .rr_need(!is.na(.rr_resume_source_file) && file.exists(.rr_resume_source_file),\n           "Source this recovery from its saved .R file so its SHA256 can be sealed")\n\n  lake_root <- file.path(CFG$results_root, "lakes", lake)\n  attempts_root <- file.path(lake_root, "attempts")\n  .rr_need(dir.exists(attempts_root), "Missing attempts directory: ", attempts_root)\n  read_sealed <- .rr_fun(R69, "r69_read_sealed")\n  if (is.null(attempt_dir)) {\n    attempts <- list.dirs(attempts_root, full.names = TRUE, recursive = FALSE)\n    eligible <- attempts[vapply(attempts, function(path) {\n      file.exists(file.path(path, "request.rds")) &&\n        !file.exists(file.path(path, "calibration_receipt.rds"))\n    }, logical(1))]\n    .rr_need(length(eligible) == 1L,\n             "Expected exactly one unsealed attempt for ", lake, "; found ",\n             length(eligible), if (length(eligible)) paste0(": ", paste(eligible, collapse = ", ")) else "")\n    attempt <- eligible[[1L]]\n  } else {\n    attempt <- attempt_dir\n  }\n  attempt <- .rr_norm(attempt)\n  .rr_need(identical(dirname(attempt), .rr_norm(attempts_root)),\n           "Attempt is outside the selected lake: ", attempt)\n  request <- read_sealed(file.path(attempt, "request.rds"))\n  .rr_need(identical(as.character(request$name), lake) &&\n             isTRUE(request$acceptance$passed) &&\n             as.integer(request$num) == 500L,\n           "Request receipt does not describe the accepted 500-member lake run")\n  live_r69 <- .rr_verify_live_request(CFG, R69, lake, request, lake_root)\n\n  run_dir <- file.path(attempt, "calibration")\n  .rr_need(dir.exists(run_dir), "Missing interrupted calibration directory: ", run_dir)\n  context_file <- file.path(run_dir, "context.rds")\n  if (!file.exists(context_file)) {\n    recovery_dirs <- rev(sort(.rr_valid_prior_evidence(attempt)))\n    restored <- FALSE\n    for (directory in recovery_dirs) {\n      ready <- file.path(directory, "context_ready.rds")\n      before_context <- file.path(directory, "context_before_finalize.rds")\n      source <- if (file.exists(ready)) ready else before_context\n      if (file.exists(source)) {\n        restored <- file.copy(source, context_file, overwrite = FALSE)\n        if (isTRUE(restored)) break\n      }\n    }\n    .rr_need(isTRUE(restored),\n             "context.rds is missing and no recovery backup could restore it")\n  }\n  .rr_need(file.exists(context_file), "Missing interrupted context.rds")\n  ctx <- readRDS(context_file)\n  .rr_need(identical(.rr_norm(ctx$result_dir), .rr_norm(run_dir)),\n           "context.rds result_dir mismatch")\n  .rr_need(identical(as.character(ctx$models), names(expected_completed)) &&\n             identical(names(expected_completed), names(expected_started)),\n           "Expected-count model order differs from the calibration context")\n  .rr_need(all(vapply(ctx$pools, nrow, integer(1)) == as.integer(request$num)),\n           "Context pools do not contain exactly 500 candidates per model")\n  .rr_need(.rr_numeric_equal(ctx$control$seed, request$seed, 0),\n           "Context seed differs from request seed")\n  get(".lw_verify_runtime", asNamespace("LakeEnsemblR"))(ctx)\n  .rr_validate_pool_files(ctx)\n\n  integrity_file <- file.path(run_dir, "context_integrity.csv")\n  receipt_file <- file.path(attempt, "calibration_receipt.rds")\n  if (file.exists(integrity_file)) {\n    checked_ctx <- .rr_fun(R69, "r69_check_calibration_integrity")(run_dir)\n    .rr_need(!is.null(checked_ctx$calibration_files),\n             "Existing integrity record is not a completed calibration")\n    summary <- .rr_read_csv(file.path(run_dir, "calibration_summary.csv"))\n    script_sha <- .rr_sha(.rr_resume_source_file)\n    evidence_dir <- .rr_new_evidence_dir(attempt)\n    return(invisible(.rr_complete_receipt(CFG, R69, attempt, checked_ctx,\n                                          request, summary, evidence_dir,\n                                          script_sha)))\n  }\n  .rr_need(!file.exists(receipt_file) && !file.exists(paste0(receipt_file, ".sha256")),\n           "A calibration receipt exists without a completed integrity record; inspect manually")\n\n  prior_evidence <- .rr_valid_prior_evidence(attempt)\n  evidence_dir <- .rr_new_evidence_dir(attempt)\n  script_sha <- .rr_sha(.rr_resume_source_file)\n  writeLines(c(paste0("script=", .rr_resume_source_file),\n               paste0("script_sha256=", script_sha),\n               paste0("LakeEnsemblR=", utils::packageVersion("LakeEnsemblR")),\n               paste0("attempt=", attempt),\n               paste0("started_UTC=", format(Sys.time(), "%Y-%m-%d %H:%M:%S", tz = "UTC"))),\n             file.path(evidence_dir, "recovery_identity.txt"), useBytes = TRUE)\n\n  before <- .rr_scan_candidates(ctx)\n  .rr_write_evidence_csv(before$index, file.path(evidence_dir, "candidate_scan_before.csv"))\n  observed_completed <- table(factor(\n    before$index$model[before$index$state == "complete"], levels = ctx$models))\n  observed_status <- table(factor(\n    before$index$model[before$index$status_exists], levels = ctx$models))\n  observed_started <- table(factor(\n    before$index$model[before$index$directory_exists], levels = ctx$models))\n  names(observed_completed) <- names(observed_status) <-\n    names(observed_started) <- ctx$models\n  current_manifest <- .rr_retained_manifest(ctx, before)\n  .rr_write_evidence_csv(current_manifest,\n                         file.path(evidence_dir, "retained_manifest_current.csv"))\n  if (!length(prior_evidence)) {\n    .rr_need(identical(as.integer(observed_status), as.integer(expected_completed)),\n             "status.csv counts differ from the reported stopped run. Observed: ",\n             paste(names(observed_status), observed_status, sep = "=", collapse = ", "))\n    .rr_need(identical(as.integer(observed_started), as.integer(expected_started)),\n             "Started counts differ from the reported stopped run. Observed: ",\n             paste(names(observed_started), observed_started, sep = "=", collapse = ", "))\n    locked_manifest <- current_manifest\n  } else {\n    prior_evidence <- sort(prior_evidence)\n    locked_manifest <- .rr_read_csv(file.path(\n      prior_evidence[[1L]], "locked_retained_manifest.csv"))\n    .rr_verify_locked_manifest(ctx, locked_manifest)\n    .rr_need(all(observed_started >= observed_completed) &&\n               all(observed_started <= as.integer(request$num)),\n             "Invalid candidate counts in repeated recovery")\n  }\n  .rr_write_evidence_csv(locked_manifest,\n                         file.path(evidence_dir, "locked_retained_manifest.csv"))\n  .rr_verify_locked_manifest(ctx, locked_manifest)\n\n  incomplete <- before$index[before$index$state == "incomplete", , drop = FALSE]\n  .rr_need(nrow(incomplete) == sum(observed_started - observed_completed),\n           "Candidate scan has inconsistent incomplete-directory counts")\n  missing_before_move <- before$index[before$index$state != "complete", , drop = FALSE]\n  if (!isTRUE(execute_missing)) {\n    .rr_write_evidence_csv(missing_before_move,\n                           file.path(evidence_dir, "missing_to_execute.csv"))\n    cat("RECOVERY_AUDIT_PASSED_NO_FILES_MOVED\\nEvidence:\\n", evidence_dir,\n        "\\nRetained validated candidates: ", sum(observed_completed),\n        "\\nCandidates still required: ", nrow(missing_before_move), "\\n", sep = "")\n    return(invisible(list(status = "audit_only", evidence_dir = evidence_dir,\n                          missing = missing_before_move, attempt_dir = attempt)))\n  }\n  stale_work <- file.path(run_dir, "_work")\n  if (dir.exists(stale_work)) {\n    stale_target <- file.path(evidence_dir, "stale_work_before_resume")\n    .rr_need(!file.exists(stale_target) && file.rename(stale_work, stale_target),\n             "Could not preserve stale calibration _work directory")\n    writeLines(c(paste0("original=", stale_work), paste0("archive=", stale_target)),\n               file.path(evidence_dir, "stale_work_archive.txt"), useBytes = TRUE)\n  }\n  archive_root <- file.path(evidence_dir, "interrupted_candidates")\n  archive_rows <- list()\n  if (nrow(incomplete)) {\n    for (i in seq_len(nrow(incomplete))) {\n      source <- incomplete$directory[i]\n      target <- file.path(archive_root, incomplete$model[i], incomplete$par_id[i])\n      .rr_need(!file.exists(target), "Archive target already exists: ", target)\n      target_parent <- dirname(target)\n      .rr_need(dir.exists(target_parent) ||\n                 dir.create(target_parent, recursive = TRUE, showWarnings = FALSE),\n               "Could not create archive parent: ", dirname(target))\n      .rr_need(file.rename(source, target),\n               "Could not preserve interrupted candidate: ", source)\n      archive_rows[[length(archive_rows) + 1L]] <- data.frame(\n        model = incomplete$model[i], par_id = incomplete$par_id[i],\n        original = source, archive = target, reason = incomplete$reason[i],\n        stringsAsFactors = FALSE)\n    }\n  }\n  archive_table <- if (length(archive_rows)) do.call(rbind, archive_rows) else\n    data.frame(model = character(), par_id = character(), original = character(),\n               archive = character(), reason = character())\n  .rr_write_evidence_csv(archive_table, file.path(evidence_dir, "archive_manifest.csv"))\n\n  after_archive <- .rr_scan_candidates(ctx)\n  missing <- after_archive$index[after_archive$index$state == "missing", , drop = FALSE]\n  .rr_need(nrow(missing) == 5L * as.integer(request$num) - sum(observed_completed),\n           "Missing-candidate count changed unexpectedly after preservation")\n  .rr_write_evidence_csv(missing, file.path(evidence_dir, "missing_to_execute.csv"))\n\n  tolerance <- if (is.null(CFG$tolerance_celsius)) 1e-6 else\n    suppressWarnings(as.numeric(CFG$tolerance_celsius))\n  .rr_need(length(tolerance) == 1L && is.finite(tolerance) && tolerance >= 0,\n           "Invalid CFG$tolerance_celsius")\n  precheck <- if (is.null(precheck_runner)) {\n    .rr_concurrency_precheck(ctx, before, ncores, evidence_dir, tolerance)\n  } else {\n    .rr_need(is.function(precheck_runner), "precheck_runner must be a function")\n    precheck_runner(ctx, before, ncores, evidence_dir, tolerance)\n  }\n  .rr_need(is.data.frame(precheck) && nrow(precheck) == 50L &&\n             all(precheck$passed %in% TRUE),\n           "The five-model 10-concurrent replay precheck did not pass")\n\n  tasks <- list()\n  for (model in ctx$models) {\n    pool <- ctx$pools[[model]]\n    ids <- missing$par_id[missing$model == model]\n    positions <- match(ids, as.character(pool$par_id))\n    .rr_need(!anyNA(positions), "Missing candidate is absent from context pool: ", model)\n    for (position in positions) {\n      id <- as.character(pool$par_id[position])\n      tasks[[length(tasks) + 1L]] <- list(\n        model = model,\n        spec = ctx$specs[[model]],\n        pool = pool[position, , drop = FALSE],\n        worker_id = paste0("resume_", model, "_", id))\n    }\n  }\n  .rr_need(length(tasks) == nrow(missing), "Task construction count mismatch")\n\n  if (!length(tasks)) {\n    results <- list()\n  } else if (is.null(worker_runner)) {\n    results <- .rr_run_lb_tasks(tasks, ctx, ncores,\n                                file.path(evidence_dir, "psock_workers.log"))\n  } else {\n    .rr_need(is.function(worker_runner), "worker_runner must be a function")\n    results <- worker_runner(tasks, ctx, ncores, evidence_dir)\n  }\n  .rr_need(is.list(results) && length(results) == length(tasks),\n           "Worker result count mismatch")\n\n  completed <- .rr_scan_candidates(ctx)\n  .rr_write_evidence_csv(completed$index, file.path(evidence_dir, "candidate_scan_after.csv"))\n  .rr_verify_locked_manifest(ctx, locked_manifest)\n  .rr_need(all(completed$index$state == "complete"),\n           "Some candidates remain incomplete; rerun this recovery script. Evidence: ", evidence_dir)\n  keys <- paste(completed$index$model, completed$index$par_id, sep = "/")\n  .rr_need(length(completed$statuses) == 5L * as.integer(request$num) &&\n             setequal(names(completed$statuses), keys),\n           "Completed status set is not exactly the locked 2500-candidate pool")\n  summary <- do.call(rbind, unname(completed$statuses))\n  summary <- summary[order(match(summary$model, ctx$models), summary$par_id), , drop = FALSE]\n  .rr_need(nrow(summary) == 5L * as.integer(request$num),\n           "Recovered calibration summary row count is not 2500")\n  best <- lapply(ctx$models, function(model) {\n    part <- summary[summary$model == model & summary$status == "ok" &\n                      is.finite(summary$objective), , drop = FALSE]\n    .rr_need(nrow(part) > 0L, "All candidates failed for ", model)\n    part[order(part$objective, part$par_id)[1L], , drop = FALSE]\n  })\n  best <- do.call(rbind, best)\n  .rr_write_or_verify_package_csv(summary,\n                                  file.path(run_dir, "calibration_summary.csv"),\n                                  evidence_dir)\n  .rr_write_or_verify_package_csv(best,\n                                  file.path(run_dir, "best_by_calibration_RMSE.csv"),\n                                  evidence_dir)\n\n  recovery_run_dir <- file.path(run_dir, "recovery")\n  .rr_need(identical(.rr_sha(.rr_resume_source_file), script_sha),\n           "Recovery script changed while the recovery was running")\n  .rr_copy_or_verify(.rr_resume_source_file,\n                     file.path(recovery_run_dir,\n                               "Resume_interrupted_calibration_9003_rev3.R"))\n  .rr_copy_or_verify(file.path(evidence_dir, "locked_retained_manifest.csv"),\n                     file.path(recovery_run_dir,\n                               "locked_retained_manifest.csv"))\n  replay_file <- file.path(evidence_dir, "five_model_10_concurrent_replay.csv")\n  .rr_copy_or_verify(replay_file,\n                     file.path(recovery_run_dir,\n                               paste0(basename(evidence_dir), "_concurrency_replay.csv")))\n  provenance <- data.frame(\n    schema = "LER9003-interrupted-calibration-resume-v1",\n    script_sha256 = script_sha,\n    scheduler = "parallel::parLapplyLB; one candidate per task; chunk.size=1",\n    ncores = ncores,\n    request_signature = request$signature,\n    live_signature = live_r69$signature,\n    acceptance_signature = request$acceptance$signature,\n    retained_validated_candidates = sum(observed_completed),\n    newly_executed_candidates = length(tasks),\n    concurrency_replay_candidates = nrow(precheck),\n    external_evidence_directory = evidence_dir,\n    completed_UTC = format(Sys.time(), "%Y-%m-%d %H:%M:%S", tz = "UTC"),\n    stringsAsFactors = FALSE)\n  .rr_write_or_verify_package_csv(\n    provenance,\n    file.path(recovery_run_dir,\n              paste0(basename(evidence_dir), "_provenance.csv")),\n    evidence_dir)\n\n  manifest <- get(".lw_file_manifest", asNamespace("LakeEnsemblR"))(run_dir)\n  manifest <- manifest[!grepl("^(_work|snapshot)/", manifest$file) &\n                         manifest$file != "context.rds", , drop = FALSE]\n  rownames(manifest) <- NULL\n  ctx$calibration_files <- manifest\n  .rr_atomic_context_replace(context_file, ctx, evidence_dir)\n  integrity <- data.frame(file = "context.rds", sha256 = .rr_sha(context_file),\n                          stringsAsFactors = FALSE)\n  .rr_write_or_verify_package_csv(integrity, integrity_file, evidence_dir)\n  checked_ctx <- .rr_fun(R69, "r69_check_calibration_integrity")(run_dir)\n  .rr_need(identical(checked_ctx$calibration_files, manifest),\n           "Final calibration manifest did not survive integrity verification")\n  selected <- .rr_complete_receipt(CFG, R69, attempt, checked_ctx, request,\n                                   summary, evidence_dir, script_sha)\n  cat("INTERRUPTED_CALIBRATION_RECOVERED\\n",\n      "Lake: ", lake, "\\n",\n      "Retained completed candidates: ", sum(observed_completed), "\\n",\n      "Newly executed candidates: ", length(tasks), "\\n",\n      "Successful candidates: ", selected$success_count, "\\n",\n      "Failed candidates: ", selected$failure_count, "\\n",\n      "Evidence: ", evidence_dir, "\\n",\n      "NEXT: evaluate_recovered_lake_only(CFG, R69, \\"", lake, "\\")\\n",\n      sep = "")\n  invisible(list(status = "complete", selected = selected,\n                 evidence_dir = evidence_dir, summary = summary, best = best))\n}\n\nevaluate_recovered_lake_only <- function(CFG, R69, lake = "akesaiqin") {\n  .rr_need(identical(as.character(utils::packageVersion("LakeEnsemblR")),\n                     "1.2.9.9003"),\n           "One-lake evaluation requires the unchanged 1.2.9.9003 environment")\n  cfg <- CFG\n  cfg$stage <- "research"\n  row <- .rr_registry_row(cfg, R69, lake)\n  live <- .rr_fun(R69, "r69_preflight")(cfg, row)\n  .rr_need(isTRUE(live$passed), "One-lake live preflight failed: ",\n           paste(live$issues, collapse = " | "))\n  request_paths <- list.files(file.path(cfg$results_root, "lakes", lake, "attempts"),\n                              pattern = "^request[.]rds$", recursive = TRUE,\n                              full.names = TRUE)\n  .rr_need(length(request_paths) >= 1L, "No sealed request exists for ", lake)\n  request_paths <- sort(request_paths, decreasing = TRUE)\n  request <- .rr_fun(R69, "r69_read_sealed")(request_paths[[1L]])\n  .rr_need(identical(live$signature, request$signature),\n           "One-lake live signature differs from the recovered request")\n  result <- .rr_fun(R69, "r69_run_lake")(cfg, row, ctx = live)\n  .rr_need(is.data.frame(result) && nrow(result) == 1L &&\n             identical(as.character(result$calibration_status), "ok") &&\n             identical(as.character(result$evaluation_status), "ok"),\n           "Recovered lake evaluation did not complete")\n  index_file <- file.path(cfg$results_root, "run_index.csv")\n  index <- .rr_read_csv(index_file)\n  position <- match(lake, index$name)\n  .rr_need(length(position) == 1L && !is.na(position),\n           "Recovered lake is absent from run_index.csv")\n  backup_dir <- file.path(cfg$results_root, "one_lake_evaluation_backups")\n  dir.create(backup_dir, recursive = TRUE, showWarnings = FALSE)\n  backup <- file.path(backup_dir,\n                      paste0("run_index_before_", lake, "_", .rr_stamp(), ".csv"))\n  .rr_need(file.copy(index_file, backup, overwrite = FALSE),\n           "Could not preserve run_index.csv before one-lake update")\n  common <- intersect(names(index), names(result))\n  index[position, common] <- result[1L, common, drop = FALSE]\n  .rr_fun(R69, "r69_write_csv")(index, index_file)\n  cat("RECOVERED_LAKE_EVALUATION_PASSED\\nLake: ", lake,\n      "\\nrun_index backup: ", backup,\n      "\\nDo not call run_stage(\\"research\\") yet; use the load-balanced remaining-lakes handoff.\\n",\n      sep = "")\n  invisible(result)\n}\n\n', 'Run_remaining67_R4ice_12.R': '# Four-model, three-parameter Simstrat experiment. Source into a NEW environment.\n# Source only defines functions; run_remaining_lakes_r4() is the entry point.\n# Keep the original CFG, R69, YAMLs, accepted receipts and old results unchanged.\n.r4_source <- local({\n  found <- as.character(unlist(lapply(sys.frames(), function(f) {\n    if (exists("ofile", envir = f, inherits = FALSE)) get("ofile", envir = f) else NULL\n  }), use.names = FALSE))\n  found <- found[basename(found) == "Run_remaining67_R4ice_12.R"]\n  if (!length(found)) NA_character_ else normalizePath(tail(found, 1L), winslash = "/", mustWork = TRUE)\n})\n.r4_models <- c("FLake", "GLM", "GOTM", "Simstrat")\n.r4_schema <- "LER4-ice-20261102-v1"\n.r4_need <- function(ok, ...) if (!isTRUE(ok)) stop(..., call. = FALSE)\n.r4_private <- function(name) get(name, asNamespace("LakeEnsemblR"))\n.r4_sha <- function(p) digest::digest(file = p, algo = "sha256", serialize = FALSE)\n.r4_hash <- function(x) digest::digest(x, algo = "sha256", serialize = TRUE)\n.r4_stamp <- function() format(Sys.time(), "%Y%m%dT%H%M%SZ", tz = "UTC")\n.r4_norm <- function(p) normalizePath(p, winslash = "/", mustWork = TRUE)\n.r4_csv <- function(x, p) {\n  if (file.exists(p)) .rr_table_equal(.rr_read_csv(p), x, p)\n  else .r4_private(".lw_write_csv")(x, p)\n  invisible(p)\n}\n.r4_save <- function(x, p) {\n  if (file.exists(p)) {\n    .r4_need(identical(.r4_read(p), x), "Existing receipt differs: ", p)\n    return(invisible(p))\n  }\n  dir.create(dirname(p), recursive = TRUE, showWarnings = FALSE)\n  tmp <- paste0(p, ".new")\n  .r4_need(!file.exists(tmp), "Interrupted receipt write; preserve/check: ", tmp)\n  saveRDS(x, tmp, version = 3L)\n  writeLines(.r4_sha(tmp), paste0(tmp, ".sha256"), useBytes = TRUE)\n  .r4_need(file.rename(tmp, p) && file.rename(paste0(tmp, ".sha256"), paste0(p, ".sha256")),\n    "Receipt installation interrupted: ", p)\n  invisible(p)\n}\n.r4_read <- function(p) {\n  seal <- paste0(p, ".sha256")\n  .r4_need(file.exists(p) && file.exists(seal) &&\n    identical(readLines(seal, warn = FALSE), .r4_sha(p)), "Receipt/seal mismatch: ", p)\n  readRDS(p)\n}\n.r4_manifest <- function(files) {\n  files <- sort(unique(.r4_norm(files)))\n  data.frame(file = files, bytes = as.numeric(file.info(files)$size),\n    sha256 = vapply(files, .r4_sha, character(1)), stringsAsFactors = FALSE, row.names = NULL)\n}\n.r4_check_files <- function(x) {\n  .r4_need(identical(.r4_manifest(x$file), x), "Recorded live input/source files changed; use a NEW results root")\n  invisible(TRUE)\n}\n.r4_archive <- function(p, evidence, name = basename(p)) {\n  if (!file.exists(p) && !dir.exists(p)) return(invisible(NULL))\n  target <- file.path(evidence, name)\n  .r4_need(!file.exists(target) && !dir.exists(target) && file.rename(p, target),\n    "Cannot preserve interrupted files: ", p)\n  invisible(target)\n}\n.r4_evidence <- function(root, prefix = "e") {\n  dir.create(root, recursive = TRUE, showWarnings = FALSE)\n  path <- tempfile(paste0(prefix, "_", .r4_stamp(), "_"), tmpdir = root)\n  .r4_need(dir.create(path), "Cannot create evidence directory")\n  .r4_norm(path)\n}\n.r4_phase <- function(path, phase, expression) {\n  start <- Sys.time(); status <- "failed"; message <- ""\n  on.exit({\n    end <- Sys.time()\n    row <- data.frame(phase = phase, start_UTC = format(start, "%Y-%m-%d %H:%M:%S", tz = "UTC"),\n      end_UTC = format(end, "%Y-%m-%d %H:%M:%S", tz = "UTC"),\n      elapsed_seconds = as.numeric(difftime(end, start, units = "secs")), status = status, message = message)\n    if (file.exists(path)) row <- rbind(.rr_read_csv(path), row)\n    tmp <- paste0(path, ".new"); if (file.exists(tmp)) unlink(tmp)\n    .r4_private(".lw_write_csv")(row, tmp)\n    if (file.exists(path)) unlink(path)\n    .r4_need(file.rename(tmp, path), "Cannot install phase log")\n  }, add = TRUE)\n  value <- tryCatch(force(expression), error = function(e) { message <<- conditionMessage(e); stop(e) })\n  status <- "ok"; value\n}\n.r4_expected_specs <- function(model) {\n  x <- data.frame(name = c("wind_speed", "swr"), type = c("met", "met"),\n    lower = c(0.5, 0.5), upper = c(2, 1.5), initial = c(1, 1), log = c(FALSE, FALSE))\n  if (model == "Simstrat") x <- rbind(x, data.frame(name = "ModelParameters/p_sw_ice", type = "model",\n    lower = 0.9, upper = 3, initial = 1, log = FALSE))\n  x\n}\n.r4_live <- function(old_CFG, config_root, row, n, ncores) {\n  name <- as.character(row$name); id <- suppressWarnings(as.numeric(row$NAME_ID))\n  .r4_need(length(id) == 1L && is.finite(id) && id > 0 && id == floor(id), "Invalid actual NAME_ID")\n  spin <- suppressWarnings(as.numeric(row$spin_up_days))\n  .r4_need(length(spin) == 1L && is.finite(spin) && spin >= 0, "Invalid preserved spin_up_days")\n  yaml <- file.path(config_root, paste0(name, "_LakeEnsemblR.yaml"))\n  cfg <- .r4_private(".lw_read_config")(yaml)\n  .r4_need(identical(as.character(cfg$location$name), name), "YAML lake name mismatch")\n  .r4_need(identical(as.character(cfg$time$start), "1979-01-01 00:00:00") &&\n    identical(as.character(cfg$time$stop), "2016-01-01 00:00:00") &&\n    isTRUE(all.equal(as.numeric(cfg$time$time_step), 3600)) &&\n    isTRUE(all.equal(as.numeric(cfg$output$time_step), 86400)), "Simulation time/steps changed")\n  .r4_need(isTRUE(cfg$input$ice$use), "Ice physics must remain enabled")\n  .r4_need(setequal(names(cfg$config_files), .r4_models) && is.null(cfg$calibration$MyLake),\n    "Expected exactly FLake, GLM, GOTM, Simstrat, with no MyLake calibration")\n  specs <- setNames(lapply(.r4_models, function(m) {\n    z <- .r4_private(".lw_specs")(cfg, m)\n    .rr_table_equal(z, .r4_expected_specs(m), paste(name, m, "design")); z\n  }), .r4_models)\n  .r4_private(".lw_check_legacy_yaml")(cfg, yaml, .r4_models)\n  win <- file.path(old_CFG$windows_dir, paste0(name, "_MODIS_windows_reviewed.csv"))\n  con <- file.path(old_CFG$contracts_dir, paste0(name, "_output_contracts.csv"))\n  contract_manifest <- file.path(dirname(old_CFG$contracts_dir), "records", "contracts69_manifest.csv")\n  approved <- .rr_read_csv(contract_manifest)\n  a <- approved[approved$name == name, , drop = FALSE]\n  .r4_need(nrow(a) == 1L && identical(as.character(a$file), basename(con)) &&\n    identical(as.character(a$sha256), .r4_sha(con)), "Existing approved time-contract manifest does not match this lake")\n  seed0 <- if (is.null(old_CFG$seed)) 20261026L else old_CFG$seed\n  seed <- as.integer((as.double(seed0) + 104729 * id) %% 2147483646 + 1)\n  control <- LakeEnsemblR::ler_window_control(win, con, seed = seed, save_series = FALSE)\n  contracts <- .r4_private(".lw_contracts")(control$contracts, .r4_models, require_confirmed = TRUE)\n  .r4_need(all(contracts$step_seconds == 86400) &&\n    !any(grepl("UNCONFIRMED|PROPOSED|TODO|待确认|未确认", contracts$meaning, ignore.case = TRUE)),\n    "Use only the existing confirmed daily output time contracts")\n  w <- control$windows; cal <- w[w$use & w$split == "calibration", , drop = FALSE]\n  val <- w[w$use & w$split == "validation", , drop = FALSE]\n  utc <- .r4_private(".lw_utc")\n  .r4_need(nrow(cal) >= 2L && nrow(val) >= 2L &&\n    all(cal$window_start >= utc("2001-01-01 00:00:00") & cal$window_end <= utc("2011-01-01 00:00:00")) &&\n    all(val$window_start >= utc("2011-01-01 00:00:00") & val$window_end <= utc("2016-01-01 00:00:00")),\n    "Calibration must use 2001-2010; validation must use 2011-2015")\n  .r4_need((!"name" %in% names(w) || all(w$name == name)) &&\n    (!"NAME_ID" %in% names(w) || all(as.numeric(w$NAME_ID) == id)), "Window lake identity mismatch")\n  refs <- c(yaml, win, con, contract_manifest, file.path(config_root, "remaining_lakes.csv"), file.path(config_root, "all_lakes.csv"))\n  add <- function(value) {\n    if (!is.character(value) || length(value) != 1L || !nzchar(value) || value == "NULL") return()\n    p <- if (grepl("^(/|[A-Za-z]:[/\\\\\\\\])", value)) value else file.path(config_root, value)\n    .r4_need(file.exists(p) && !dir.exists(p), "Missing live input: ", p)\n    refs <<- c(refs, p)\n  }\n  walk <- function(x) {\n    if (!is.list(x)) return()\n    for (key in names(x)) if (key == "file") add(x[[key]]) else if (is.list(x[[key]])) walk(x[[key]])\n  }\n  add(cfg$location$hypsograph)\n  walk(cfg$input)\n  walk(cfg$observations)\n  for (sec in c("inflows", "outflows")) {\n    if (!is.null(cfg[[sec]]$file)) add(cfg[[sec]]$file)\n  }\n  kw <- cfg$input$light$Kw\n  if (is.list(kw)) for (z in kw) add(z) else add(kw)\n  for (m in .r4_models) {\n    d <- file.path(config_root, m)\n    .r4_need(dir.exists(d), "Native template folder missing: ", d)\n    rel <- list.files(d, recursive = TRUE, full.names = TRUE, all.files = TRUE, no.. = TRUE)\n    refs <- c(refs, rel[!grepl("/output/", rel)])\n  }\n  generated <- .rr_read_csv(file.path(config_root, "_generated_sha256.csv"))\n  .r4_need(all(c("relative_path", "sha256") %in% names(generated)), "Missing generated configuration manifest")\n  for (i in seq_len(nrow(generated))) .r4_need(identical(.r4_sha(file.path(config_root, generated$relative_path[i])),\n    as.character(generated$sha256[i])), "Generated configuration changed: ", generated$relative_path[i])\n  source <- .r4_source\n  .r4_need(length(source) == 1L && !is.na(source) && file.exists(source), "Source this saved R script before running")\n  helper <- .rr_resume_source_file\n  .r4_need(length(helper) == 1L && !is.na(helper) && file.exists(helper) && identical(.r4_sha(helper),\n    "74cfbe6a90da76df42c80b168edb45d4a1e78ba08dceac4976ffff84516e5afa"), "Load the preserved rev3 recovery helper into this separate R4 environment")\n  sources <- c(source, helper, file.path(config_root, "_generated_sha256.csv"))\n  runtime <- .r4_private(".lw_package_state")(c("LakeEnsemblR", "FLakeR", "GLM3r", "GOTMr", "SimstratR", "digest", "yaml"))\n  funcs <- sort(c(grep("^\\\\.lw_", ls(asNamespace("LakeEnsemblR"), all.names = TRUE), value = TRUE),\n    "export_config", "export_init_cond", "input_config_value", "evaluate_window_best", "calc_fit_window"))\n  code <- vapply(funcs, function(nm) .r4_private(".lw_function_hash")(.r4_private(nm)), character(1))\n  payload <- list(schema = .r4_schema, name = name, NAME_ID = id, num = as.integer(n),\n    ncores = as.integer(ncores), models = .r4_models, seed = seed, spin_up_days = spin,\n    specs = specs, live_files = .r4_manifest(refs), sources = .r4_manifest(sources),\n    runtime = runtime, runtime_functions = code,\n    helper_functions = vapply(c(".rr_validate_candidate", ".rr_scan_candidates", ".rr_run_lb_tasks", ".rr_atomic_context_replace",\n      ".rr_retained_manifest", ".rr_verify_locked_manifest", ".rr_table_equal"), function(nm)\n      .r4_private(".lw_function_hash")(get(nm, mode = "function", inherits = TRUE)), character(1)), controls = control,\n    min_success_fraction = 0.95, tolerance_celsius = 1e-6)\n  list(payload = payload, signature = .r4_hash(payload), config_file = basename(yaml),\n    folder = .r4_norm(config_root), control = control, spin_up = spin)\n}\n.r4_new_context <- function(live, run) {\n  .r4_check_files(live$payload$live_files)\n  .r4_check_files(live$payload$sources)\n  ctx <- .r4_private(".lw_context")(live$config_file, live$folder, .r4_models,\n    live$control, run, live$spin_up)\n  ctx$specs <- live$payload$specs\n  ctx$pools <- setNames(lapply(.r4_models, function(m) .r4_private(".lw_lhc")(\n    ctx$specs[[m]], live$payload$num, .r4_private(".lw_seed")(live$payload$seed, m, "LHC"))), .r4_models)\n  for (m in .r4_models) {\n    .r4_csv(ctx$specs[[m]], file.path(run, paste0("spec_", m, ".csv")))\n    .r4_csv(ctx$pools[[m]], file.path(run, paste0("params_", m, ".csv")))\n  }\n  .r4_private(".lw_select")(ctx$windows, ctx$control$validation_split, ctx$allowed_start, ctx$allowed_end)\n  .r4_need(file.copy(.r4_source, file.path(run, "workflow_R4ice.R")), "Cannot retain workflow source")\n  .r4_need(file.copy(.rr_resume_source_file, file.path(run, "recovery_helper_rev3.R")), "Cannot retain helper source")\n  .r4_csv(live$payload$sources, file.path(run, "workflow_sources.csv"))\n  .r4_csv(live$payload$live_files, file.path(run, "live_input_files.csv"))\n  ctx$r4_signature <- live$signature; ctx$r4_schema <- .r4_schema\n  saveRDS(ctx, file.path(run, "context.rds"), version = 3L)\n  .r4_check_files(live$payload$live_files)\n  .r4_check_files(live$payload$sources)\n  ctx\n}\n.r4_export_check <- function(ctx, root) {\n  .r4_need(dir.create(root, recursive = TRUE), "Cannot create new native export audit")\n  .r4_private(".lw_copy_tree")(ctx$snapshot, root)\n  old <- getwd(); on.exit(setwd(old), add = TRUE); setwd(root)\n  LakeEnsemblR::export_config(ctx$config_file, model = .r4_models, folder = ".")\n  ini <- file.path("Simstrat", "init_cond.dat")\n  lines <- readLines(ini, warn = FALSE)\n  header <- trimws(gsub("[[:space:]]*\\\\[.*$", "", strsplit(lines[1L], "\\t", fixed = TRUE)[[1L]]))\n  values <- as.matrix(utils::read.table(ini, skip = 1L, sep = "\\t", header = FALSE))\n  .r4_need(identical(header, c("Depth", "U", "V", "T", "S", "k", "eps")) &&\n    ncol(values) == 7L && all(is.finite(values)) && all(values[, 5L] == 0) &&\n    all(values[, 6L] > 0 & values[, 7L] > 0), "Simstrat seven-column initialization audit failed")\n  spec <- ctx$specs$Simstrat\n  for (v in c(0.9, 1, 3)) {\n    row <- as.data.frame(as.list(setNames(spec$initial, spec$name)), check.names = FALSE)\n    row[["ModelParameters/p_sw_ice"]] <- v\n    .r4_private(".lw_apply_native")(ctx$base, "Simstrat", spec, row)\n    text <- readLines(ctx$base$config_files$Simstrat, warn = FALSE)\n    key <- grep(\'"p_sw_ice"[[:space:]]*:\', text, value = TRUE)\n    value <- suppressWarnings(as.numeric(sub(\'.*"p_sw_ice"[[:space:]]*:[[:space:]]*([^,!}]+).*\', \'\\\\1\', key)))\n    .r4_need(length(value) == 1L && isTRUE(all.equal(value, v, tolerance = 0)),\n      "Native p_sw_ice write/read failed")\n    file.copy(ctx$base$config_files$Simstrat, paste0("p_sw_ice_", v, ".json"))\n  }\n  invisible(TRUE)\n}\n.r4_tasks <- function(ctx, todo = NULL) {\n  tasks <- list(); longest <- max(vapply(ctx$pools, nrow, integer(1)))\n  for (i in seq_len(longest)) for (m in ctx$models) {\n    if (i > nrow(ctx$pools[[m]])) next\n    p <- ctx$pools[[m]][i, , drop = FALSE]\n    if (!is.null(todo) && !paste(m, p$par_id, sep = "/") %in% todo) next\n    tasks[[length(tasks) + 1L]] <- list(model = m, spec = ctx$specs[[m]], pool = p,\n      worker_id = sprintf("j%06d", length(tasks) + 1L))\n  }\n  tasks\n}\n.r4_preflight <- function(ctx, lake_root, signature, ncores) {\n  receipt <- file.path(lake_root, "preflight.rds")\n  if (file.exists(receipt)) {\n    old <- .r4_read(receipt)\n    .r4_need(identical(old$signature, signature) && isTRUE(old$passed), "New-design preflight signature differs")\n    .r4_private(".lw_check_manifest")(file.path(lake_root, "v"), old$files, "New-design preflight")\n    return(invisible(old))\n  }\n  root <- file.path(lake_root, "v")\n  if (dir.exists(root)) .r4_archive(root, .r4_evidence(file.path(lake_root, "e")), "preflight_incomplete")\n  dir.create(root, recursive = TRUE)\n  .r4_export_check(ctx, file.path(root, "x"))\n  serial <- ctx; serial$result_dir <- file.path(root, "s"); dir.create(serial$result_dir)\n  serial$control$save_series <- TRUE\n  serial$pools <- setNames(lapply(ctx$models, function(m) {\n    p <- as.data.frame(as.list(setNames(ctx$specs[[m]]$initial, ctx$specs[[m]]$name)), check.names = FALSE)\n    if (m == "Simstrat") {\n      p <- p[rep(1L, 3L), , drop = FALSE]; p[["ModelParameters/p_sw_ice"]] <- c(0.9, 1, 3)\n    }\n    p$par_id <- sprintf("v%02d", seq_len(nrow(p))); rownames(p) <- NULL; p\n  }), ctx$models)\n  serial_out <- lapply(.r4_tasks(serial), .r4_private(".lw_worker"), ctx = serial)\n  baseline <- do.call(rbind, serial_out)\n  .r4_csv(baseline, file.path(root, "serial_status.csv"))\n  .r4_need(all(baseline$status == "ok"), "New-design baseline engine failed; inspect preflight v/s/calibration")\n  concurrent <- serial; concurrent$models <- "Simstrat"\n  concurrent$pools <- serial$pools["Simstrat"]; concurrent$specs <- serial$specs["Simstrat"]\n  concurrent$result_dir <- file.path(root, "p"); dir.create(concurrent$result_dir)\n  parallel_out <- .rr_run_lb_tasks(.r4_tasks(concurrent), concurrent, ncores, file.path(root, "workers.log"))\n  .r4_csv(do.call(rbind, parallel_out), file.path(root, "parallel_status.csv"))\n  for (id in serial$pools$Simstrat$par_id) {\n    a <- file.path(serial$result_dir, "calibration", "Simstrat", id)\n    b <- file.path(concurrent$result_dir, "calibration", "Simstrat", id)\n    .r4_need(all(.rr_read_csv(file.path(b, "status.csv"))$status == "ok"), "New Simstrat concurrent case failed")\n    .rr_table_equal(.rr_read_csv(file.path(b, "parameters.csv")), .rr_read_csv(file.path(a, "parameters.csv")), "same parameter replay")\n    x <- readRDS(file.path(a, "native_surface.rds")); y <- readRDS(file.path(b, "native_surface.rds"))\n    .r4_need(identical(as.numeric(x$datetime), as.numeric(y$datetime)) &&\n      all(is.finite(x$temperature)) && all(is.finite(y$temperature)) &&\n      all(abs(x$temperature - y$temperature) <= 1e-6), "Simstrat serial/parallel native results differ")\n    .rr_table_equal(.rr_read_csv(file.path(b, "pairs.csv")), .rr_read_csv(file.path(a, "pairs.csv")), "same candidate pairs", 1e-6)\n  }\n  value <- list(schema = .r4_schema, signature = signature, passed = TRUE,\n    models = .r4_models, native_ice_values = c(0.9, 1, 3), changed_model_parallel_cases = 3L,\n    files = .r4_private(".lw_file_manifest")(root))\n  .r4_save(value, receipt)\n  cat("NEW_DESIGN_PREFLIGHT_AND_SIMSTRAT_REPLAY_PASSED\\nLake:", ctx$base$location$name, "\\n")\n  invisible(value)\n}\n.r4_cal_integrity <- function(run) {\n  seal <- .rr_read_csv(file.path(run, "context_integrity.csv"))\n  .r4_need(nrow(seal) == 1L && identical(as.character(seal$file), "context.rds") &&\n    identical(as.character(seal$sha256), .r4_sha(file.path(run, "context.rds"))), "Calibration context seal mismatch")\n  ctx <- readRDS(file.path(run, "context.rds"))\n  .r4_need(identical(ctx$result_dir, .r4_norm(run)) && identical(ctx$r4_schema, .r4_schema), "Invalid/moved new-design context")\n  .r4_private(".lw_verify_runtime")(ctx)\n  .r4_need(!is.null(ctx$calibration_files), "Calibration not finalized")\n  for (i in seq_len(nrow(ctx$calibration_files))) .r4_need(identical(\n    .r4_sha(file.path(run, ctx$calibration_files$file[i])), ctx$calibration_files$sha256[i]), "Calibration artifact changed")\n  ctx\n}\n.r4_load_context <- function(run, root, signature) {\n  p <- file.path(run, "context.rds")\n  if (!file.exists(p)) {\n    files <- list.files(file.path(root, "e"), pattern = "^context_ready[.]rds$", recursive = TRUE, full.names = TRUE)\n    if (!length(files)) files <- list.files(file.path(root, "e"), pattern = "^context_before_finalize[.]rds$", recursive = TRUE, full.names = TRUE)\n    .r4_need(length(files) == 1L, "Missing context; cannot uniquely recover its atomic-write evidence")\n    saved <- readRDS(files)\n    .r4_need(identical(saved$r4_signature, signature) && identical(saved$models, .r4_models) &&\n      identical(saved$result_dir, .r4_norm(run)), "Context recovery identity mismatch")\n    .r4_private(".lw_verify_runtime")(saved)\n    if (!is.null(saved$calibration_files)) for (i in seq_len(nrow(saved$calibration_files))) .r4_need(identical(\n      .r4_sha(file.path(run, saved$calibration_files$file[i])), saved$calibration_files$sha256[i]), "Context recovery artifact mismatch")\n    .r4_need(file.copy(files, p, overwrite = FALSE), "Cannot restore context from preserved finalization evidence")\n  }\n  readRDS(p)\n}\n.r4_finalize <- function(ctx, scan, evidence) {\n  .r4_need(all(scan$index$state == "complete"), "Incomplete calibration after execution")\n  all <- do.call(rbind, scan$statuses)\n  all <- all[order(match(all$model, ctx$models), all$par_id), , drop = FALSE]; rownames(all) <- NULL\n  fractions <- do.call(rbind, lapply(ctx$models, function(m) {\n    p <- all[all$model == m, , drop = FALSE]\n    data.frame(model = m, requested = nrow(ctx$pools[[m]]), completed = nrow(p),\n      successful = sum(p$status == "ok" & is.finite(p$objective)), failed = sum(p$status == "failed"),\n      success_fraction = mean(p$status == "ok" & is.finite(p$objective)))\n  }))\n  .r4_csv(fractions, file.path(evidence, "success_fractions.csv"))\n  .r4_need(all(fractions$success_fraction >= 0.95), "At least one model has <95% successful candidates; inspect recorded failures")\n  best <- do.call(rbind, lapply(ctx$models, function(m) {\n    p <- all[all$model == m & all$status == "ok" & is.finite(all$objective), , drop = FALSE]\n    p[order(p$objective, p$par_id)[1L], , drop = FALSE]\n  })); rownames(best) <- NULL\n  .r4_csv(all, file.path(ctx$result_dir, "calibration_summary.csv"))\n  .r4_csv(best, file.path(ctx$result_dir, "best_by_calibration_RMSE.csv"))\n  .r4_csv(fractions, file.path(ctx$result_dir, "candidate_success_fractions.csv"))\n  manifest <- .r4_private(".lw_file_manifest")(ctx$result_dir)\n  manifest <- manifest[!grepl("^(_work|snapshot)/", manifest$file) &\n    !manifest$file %in% c("context.rds", "context_integrity.csv"), , drop = FALSE]; rownames(manifest) <- NULL\n  ctx$calibration_files <- manifest\n  .rr_atomic_context_replace(file.path(ctx$result_dir, "context.rds"), ctx, evidence)\n  .r4_csv(data.frame(file = "context.rds", sha256 = .r4_sha(file.path(ctx$result_dir, "context.rds"))),\n    file.path(ctx$result_dir, "context_integrity.csv"))\n  .r4_cal_integrity(ctx$result_dir)\n}\n.r4_calibrate <- function(ctx, root, live, ncores) {\n  receipt <- file.path(root, "calibration.rds")\n  if (file.exists(receipt)) {\n    old <- .r4_read(receipt)\n    .r4_need(identical(old$signature, live$signature) && isTRUE(old$passed), "Completed calibration request changed")\n    return(.r4_cal_integrity(ctx$result_dir))\n  }\n  if (file.exists(file.path(ctx$result_dir, "context_integrity.csv"))) {\n    ctx <- .r4_cal_integrity(ctx$result_dir)\n  } else {\n    evidence <- .r4_evidence(file.path(root, "e"), "cal")\n    scan <- .rr_scan_candidates(ctx)\n    .r4_csv(scan$index, file.path(evidence, "candidate_scan_before.csv"))\n    locked <- .rr_retained_manifest(ctx, scan)\n    .r4_csv(locked, file.path(evidence, "retained_files.csv"))\n    incomplete <- scan$index[scan$index$state == "incomplete", , drop = FALSE]\n    for (i in seq_len(nrow(incomplete))) .r4_archive(incomplete$directory[i], evidence,\n      paste(incomplete$model[i], incomplete$par_id[i], sep = "_"))\n    .r4_archive(file.path(ctx$result_dir, "_work"), evidence, "interrupted_work")\n    todo <- scan$index[scan$index$state != "complete", , drop = FALSE]\n    tasks <- .r4_tasks(ctx, paste(todo$model, todo$par_id, sep = "/"))\n    cat("Lake:", live$payload$name, "Retained:", sum(scan$index$state == "complete"),\n      "Candidates to execute:", length(tasks), "\\n")\n    .r4_phase(file.path(root, "phase_times.csv"), "calibration_candidates", {\n      .rr_run_lb_tasks(tasks, ctx, ncores, file.path(evidence, "workers.log"))\n    })\n    if (nrow(locked)) .rr_verify_locked_manifest(ctx, locked)\n    after <- .rr_scan_candidates(ctx)\n    .r4_csv(after$index, file.path(evidence, "candidate_scan_after.csv"))\n    ctx <- .r4_finalize(ctx, after, evidence)\n  }\n  summary <- .rr_read_csv(file.path(ctx$result_dir, "calibration_summary.csv"))\n  value <- list(schema = .r4_schema, signature = live$signature, passed = TRUE,\n    run_dir = ctx$result_dir, num = live$payload$num, models = ctx$models,\n    success_count = sum(summary$status == "ok"), failure_count = sum(summary$status == "failed"),\n    context_sha256 = .r4_sha(file.path(ctx$result_dir, "context.rds")))\n  .r4_save(value, receipt); ctx\n}\n.r4_verify_final <- function(run) {\n  ctx <- .r4_cal_integrity(run)\n  metrics <- .rr_read_csv(file.path(run, "final_metrics.csv"))\n  expected <- as.vector(outer(ctx$models, c("calibration", "validation"), paste, sep = "/"))\n  .r4_need(nrow(metrics) == 8L && !anyDuplicated(paste(metrics$model, metrics$split, sep = "/")) &&\n    setequal(paste(metrics$model, metrics$split, sep = "/"), expected), "Incomplete four-model final metrics")\n  best <- .rr_read_csv(file.path(run, "best_by_calibration_RMSE.csv"))\n  val <- .r4_private(".lw_select")(ctx$windows, ctx$control$validation_split, ctx$allowed_start, ctx$allowed_end)\n  for (m in ctx$models) {\n    id <- best$par_id[best$model == m]; .r4_need(length(id) == 1L, "Missing unique selected candidate")\n    directory <- file.path(run, "final", m, id)\n    native <- readRDS(file.path(directory, "native_surface.rds"))\n    con <- ctx$contracts[ctx$contracts$model == m, , drop = FALSE]\n    for (split in c("calibration", "validation")) {\n      windows <- if (split == "calibration") ctx$cal else val\n      fit <- LakeEnsemblR::calc_fit_window(native, windows, con)\n      recorded <- .rr_read_csv(file.path(directory, paste0("pairs_", split, ".csv")))\n      .rr_table_equal(recorded, fit$pairs, paste(m, split, "pairs"), 1e-9)\n      row <- metrics[metrics$model == m & metrics$split == split, , drop = FALSE]\n      for (key in names(fit$metrics)) .r4_need(.rr_numeric_equal(row[[key]], fit$metrics[[key]], 1e-9),\n        "Recalculated final metrics differ: ", m, "/", split, "/", key)\n      if (split == "calibration") {\n        old <- .rr_read_csv(file.path(run, "calibration", m, id, "pairs.csv"))\n        old <- old[match(recorded$window_id, old$window_id), , drop = FALSE]\n        .r4_need(!anyNA(old$window_id) && all(abs(old$predicted - recorded$predicted) <= 1e-6),\n          "Selected calibration/rerun mismatch")\n      }\n    }\n  }\n  list(metrics = metrics, files = .r4_private(".lw_file_manifest")(file.path(run, "final")))\n}\n.r4_evaluate <- function(ctx, root, signature) {\n  receipt <- file.path(root, "evaluation.rds")\n  if (file.exists(receipt)) {\n    old <- .r4_read(receipt)\n    .r4_need(isTRUE(old$passed) && identical(old$signature, signature), "Evaluation signature mismatch")\n    verify <- .r4_verify_final(ctx$result_dir)\n    .r4_need(identical(old$final_files, verify$files) && identical(old$metrics_sha256,\n      .r4_sha(file.path(ctx$result_dir, "final_metrics.csv"))), "Final outputs changed")\n    return(verify$metrics)\n  }\n  if (!file.exists(file.path(ctx$result_dir, "final_metrics.csv"))) {\n    if (dir.exists(file.path(ctx$result_dir, "final")) || dir.exists(file.path(ctx$result_dir, "_final_work"))) {\n      evidence <- .r4_evidence(file.path(root, "e"), "eval")\n      .r4_archive(file.path(ctx$result_dir, "final"), evidence, "interrupted_final")\n      .r4_archive(file.path(ctx$result_dir, "_final_work"), evidence, "interrupted_work")\n    }\n    .r4_phase(file.path(root, "phase_times.csv"), "evaluate_selected", {\n      LakeEnsemblR::evaluate_window_best(ctx$result_dir, tolerance_celsius = 1e-6)\n    })\n  }\n  verify <- .r4_verify_final(ctx$result_dir)\n  .r4_save(list(schema = .r4_schema, signature = signature, passed = TRUE,\n    metrics_sha256 = .r4_sha(file.path(ctx$result_dir, "final_metrics.csv")), final_files = verify$files), receipt)\n  verify$metrics\n}\n.r4_report <- function(ctx, root) {\n  out <- file.path(root, "report")\n  receipt <- file.path(root, "report.rds")\n  if (file.exists(receipt)) {\n    old <- .r4_read(receipt)\n    .r4_need(identical(old$signature, ctx$r4_signature), "Report signature differs")\n    .r4_private(".lw_check_manifest")(out, old$files, "Report")\n    return(invisible(out))\n  }\n  if (dir.exists(out)) .r4_archive(out, .r4_evidence(file.path(root, "e"), "rep"), "interrupted_report")\n  dir.create(out)\n  metrics <- .rr_read_csv(file.path(ctx$result_dir, "final_metrics.csv"))\n  .r4_csv(metrics, file.path(out, "model_metrics.csv"))\n  best <- .rr_read_csv(file.path(ctx$result_dir, "best_by_calibration_RMSE.csv"))\n  pars <- do.call(rbind, lapply(ctx$models, function(m) {\n    p <- ctx$pools[[m]][ctx$pools[[m]]$par_id == best$par_id[best$model == m], , drop = FALSE]\n    data.frame(model = m, par_id = p$par_id, parameter = ctx$specs[[m]]$name,\n      value = as.numeric(p[1L, ctx$specs[[m]]$name, drop = TRUE]), row.names = NULL)\n  }))\n  .r4_csv(pars, file.path(out, "selected_parameters.csv"))\n  all <- .rr_read_csv(file.path(ctx$result_dir, "calibration_summary.csv"))\n  response <- do.call(rbind, lapply(ctx$models, function(m) {\n    p <- ctx$pools[[m]]; z <- all[all$model == m, ]; z <- z[match(p$par_id, z$par_id), ]\n    do.call(rbind, lapply(ctx$specs[[m]]$name, function(k) data.frame(model = m, par_id = p$par_id,\n      parameter = k, value = p[[k]], status = z$status, RMSE = z$RMSE)))\n  }))\n  .r4_csv(response, file.path(out, "parameter_response.csv"))\n  pairs <- list(); ensemble_metrics <- list()\n  for (split in c("calibration", "validation")) {\n    pp <- lapply(ctx$models, function(m) {\n      p <- file.path(ctx$result_dir, "final", m, best$par_id[best$model == m], paste0("pairs_", split, ".csv"))\n      x <- .rr_read_csv(p); x$model <- m; x$split <- split; x\n    })\n    x <- pp[[1L]]\n    predicted <- sapply(pp, function(z) {\n      .r4_need(setequal(z$window_id, x$window_id), "Ensemble windows differ")\n      z$predicted[match(x$window_id, z$window_id)]\n    })\n    x$predicted <- rowMeans(predicted); x$model <- "Ensemble4"\n    x$residual <- x$predicted - x$observed\n    pairs[[split]] <- do.call(rbind, c(pp, list(x)))\n    err <- x$predicted - x$observed\n    ensemble_metrics[[split]] <- data.frame(model = "Ensemble4", split = split, N = nrow(x),\n      RMSE = sqrt(mean(err^2)), MAE = mean(abs(err)), bias = mean(err),\n      NSE = if (sum((x$observed - mean(x$observed))^2) > 0) 1 - sum(err^2) / sum((x$observed - mean(x$observed))^2) else NA_real_,\n      r = if (stats::sd(x$observed) > 0 && stats::sd(x$predicted) > 0) stats::cor(x$observed, x$predicted) else NA_real_)\n  }\n  .r4_csv(do.call(rbind, pairs), file.path(out, "pairs_models_ensemble4.csv"))\n  .r4_csv(do.call(rbind, ensemble_metrics), file.path(out, "ensemble4_metrics.csv"))\n  grDevices::png(file.path(out, "RMSE_by_split.png"), width = 1500, height = 800, res = 150)\n  oldpar <- graphics::par(mfrow = c(1, 2), mar = c(6, 5, 3, 1))\n  on.exit({if (grDevices::dev.cur() > 1L) grDevices::dev.off()}, add = TRUE)\n  for (split in c("calibration", "validation")) {\n    x <- metrics[metrics$split == split, ]; x <- x[match(ctx$models, x$model), ]\n    graphics::barplot(x$RMSE, names.arg = x$model, las = 2, ylab = "RMSE (deg C)", main = split,\n      col = c("#3686A0", "#E3A437", "#4D9D70", "#A06FAB"))\n  }\n  graphics::par(oldpar); grDevices::dev.off()\n  writeLines(c("Equal-window RMSE; selection uses calibration (2001-2010) only.",\n    "Validation: 2011-2015; four-model ensemble uses fixed equal weights.",\n    "Parameter-response scatter data are not delta/S1 global sensitivity indices.",\n    "LHC parameters and all success/failure statuses are retained for a separate sensitivity analysis."),\n    file.path(out, "READ_ME.txt"))\n  .r4_save(list(schema = .r4_schema, signature = ctx$r4_signature,\n    files = .r4_private(".lw_file_manifest")(out)), receipt)\n  invisible(out)\n}\n\n# Session-only PSOCK transport setup. Does not edit model/workflow functions.\nr4_prepare_local_parallel <- function(ncores = 10L) {\n  ns <- asNamespace("parallel")\n  set_defaults <- get("setDefaultClusterOptions", ns)\n  option_env <- get("defaultClusterOptions", ns)\n  old <- lapply(c("port", "master", "setup_strategy"), function(n) get(n, option_env))\n  names(old) <- c("port", "master", "setup_strategy")\n  port <- NA_integer_\n  candidates <- 22000L + ((Sys.getpid() + (0:63) * 997L) %% 20000L)\n  for (candidate in candidates) {\n    listener <- try(suppressWarnings(serverSocket(candidate)), silent = TRUE)\n    if (inherits(listener, "try-error")) next\n    close(listener)\n    port <- as.integer(candidate)\n    break\n  }\n  if (is.na(port)) stop("No usable local PSOCK port found; no models started.")\n  set_defaults(port = port, master = "127.0.0.1", setup_strategy = "parallel")\n  cluster <- NULL\n  passed <- FALSE\n  on.exit({\n    if (!is.null(cluster)) try(parallel::stopCluster(cluster), silent = TRUE)\n    if (!passed) do.call(set_defaults, old)\n  }, add = TRUE)\n  log_file <- file.path(tempdir(), paste0("R4_psock_", port, ".log"))\n  cat("Testing local PSOCK port:", port, "\\nWorker log:", log_file, "\\n")\n  cluster <- parallel::makePSOCKcluster(\n    ncores, outfile = log_file, setup_timeout = 30L\n  )\n  pids <- unlist(parallel::clusterCall(cluster, Sys.getpid), use.names = FALSE)\n  stopifnot(length(pids) == ncores, length(unique(pids)) == ncores)\n  parallel::stopCluster(cluster)\n  cluster <- NULL\n  passed <- TRUE\n  cat("R4_LOCAL_PSOCK_TEST_PASSED\\nWorkers:", ncores, " Port:", port, "\\n")\n  invisible(port)\n}\n\n# The two completed lakes are verified read-only and excluded from this batch.\n.r67_completed <- c("akesaiqin", "namco")\n.r67_default_config <- "C:/Geography/R/dissertation_LakeEnsemblR/yaml/r4_ice_20261103_core12"\n\n.r67_rows <- function(config_root) {\n  check <- function(x, n, label) {\n    .r4_need(is.data.frame(x) && nrow(x) == n && all(c("name", "NAME_ID", "spin_up_days") %in% names(x)),\n      label, " must have ", n, " rows and name, NAME_ID, spin_up_days")\n    x <- x[c("name", "NAME_ID", "spin_up_days")]\n    x$name <- as.character(x$name)\n    ids <- suppressWarnings(as.numeric(x$NAME_ID))\n    spin <- suppressWarnings(as.numeric(x$spin_up_days))\n    .r4_need(!anyNA(x$name) && !anyDuplicated(tolower(x$name)) &&\n      all(grepl("^[A-Za-z0-9][A-Za-z0-9_-]*$", x$name)) &&\n      all(is.finite(ids) & ids > 0 & ids == floor(ids)) && !anyDuplicated(ids) &&\n      all(is.finite(spin) & spin >= 0), "Invalid registry identity/spin-up: ", label)\n    x$NAME_ID <- as.integer(ids); x$spin_up_days <- spin\n    x <- x[order(x$name), , drop = FALSE]; rownames(x) <- NULL\n    x\n  }\n  all <- check(.rr_read_csv(file.path(config_root, "all_lakes.csv")), 69L, "all_lakes.csv")\n  remaining <- check(.rr_read_csv(file.path(config_root, "remaining_lakes.csv")), 67L, "remaining_lakes.csv")\n  expected <- all[!all$name %in% .r67_completed, , drop = FALSE]; rownames(expected) <- NULL\n  .r4_need(identical(remaining, expected) && !any(remaining$name %in% .r67_completed),\n    "Remaining registry must contain exactly the 67 other lakes, with original NAME_ID and spin-up")\n  yaml_names <- sub("_LakeEnsemblR[.]yaml$", "", list.files(config_root, pattern = "_LakeEnsemblR[.]yaml$"))\n  .r4_need(setequal(yaml_names, all$name), "69 generated YAMLs and all_lakes.csv disagree")\n  manifest <- .rr_read_csv(file.path(config_root, "_yaml_manifest.csv"))\n  k <- match(all$name, manifest$name)\n  .r4_need(nrow(manifest) == 69L && !anyNA(k) && !anyDuplicated(manifest$name) &&\n    all(as.numeric(manifest$spin_up_days[k]) == all$spin_up_days), "YAML spin-up manifest and registry disagree")\n  list(all = all, remaining = remaining)\n}\n\n.r67_shared_check <- function(config_root, ncores, num) {\n  .r4_need(identical(as.integer(ncores), 12L) && ncores == 12 &&\n    identical(as.integer(num), 500L) && num == 500, "This release uses exactly N=500 and 12 workers")\n  .r4_private(".lw_initialize_runtime")()\n  .r4_need(identical(as.character(utils::packageVersion("LakeEnsemblR")), "1.2.9.9003"),\n    "Use the existing isolated LakeEnsemblR 1.2.9.9003 library")\n  .r4_need(length(.r4_source) == 1L && !is.na(.r4_source) && file.exists(.r4_source),\n    "Load the saved remaining-67 runner and record its actual source path")\n  .r4_need(length(.rr_resume_source_file) == 1L && !is.na(.rr_resume_source_file) &&\n    file.exists(.rr_resume_source_file) && identical(.r4_sha(.rr_resume_source_file),\n    "74cfbe6a90da76df42c80b168edb45d4a1e78ba08dceac4976ffff84516e5afa"),\n    "The unchanged rev3 recovery helper must be loaded from its actual file")\n  generated <- .rr_read_csv(file.path(config_root, "_generated_sha256.csv"))\n  .r4_need(all(c("relative_path", "sha256") %in% names(generated)) &&\n    all(c("all_lakes.csv", "remaining_lakes.csv") %in% generated$relative_path),\n    "Use the 20261103 configuration manifest containing both 69/67 registries")\n  for (i in seq_len(nrow(generated))) .r4_need(identical(\n    .r4_sha(file.path(config_root, generated$relative_path[i])), as.character(generated$sha256[i])),\n    "Generated shared configuration changed: ", generated$relative_path[i])\n  .r67_rows(config_root)\n}\n\n.r67_semantic_config <- function(cfg, folder) {\n  file_value <- function(value) {\n    if (is.null(value) || !is.character(value) || length(value) != 1L || value %in% c("", "NULL")) return(value)\n    path <- if (grepl("^(/|[A-Za-z]:[/\\\\\\\\])", value)) value else file.path(folder, value)\n    .r4_need(file.exists(path) && !dir.exists(path), "Missing configuration comparison input: ", path)\n    paste0("sha256:", .r4_sha(path))\n  }\n  walk <- function(x) {\n    if (!is.list(x)) return(x)\n    for (nm in names(x)) if (nm == "file") x[nm] <- list(file_value(x[[nm]])) else\n      if (is.list(x[[nm]])) x[[nm]] <- walk(x[[nm]])\n    x\n  }\n  cfg$location$hypsograph <- file_value(cfg$location$hypsograph)\n  for (sec in c("input", "observations", "inflows", "outflows")) cfg[sec] <- list(walk(cfg[[sec]]))\n  kw <- cfg$input$light$Kw\n  if (is.list(kw)) cfg$input$light$Kw <- lapply(kw, file_value) else cfg$input$light$Kw <- file_value(kw)\n  cfg$config_files <- lapply(cfg$config_files, file_value)\n  canonical <- function(x) {\n    if (!is.list(x)) return(x)\n    if (!is.null(names(x))) x <- x[order(names(x))]\n    lapply(x, canonical)\n  }\n  canonical(cfg)\n}\n\n.r67_verify_completed_two <- function(completed_root, config_root, old_CFG, all_rows) {\n  completed_root <- .r4_norm(completed_root)\n  rows <- list()\n  for (name in .r67_completed) {\n    root <- file.path(completed_root, name); run <- file.path(root, "c")\n    request <- .r4_read(file.path(root, "request.rds"))\n    calibration <- .r4_read(file.path(root, "calibration.rds"))\n    evaluation <- .r4_read(file.path(root, "evaluation.rds"))\n    report <- .r4_read(file.path(root, "report.rds"))\n    row <- all_rows[all_rows$name == name, , drop = FALSE]\n    seed0 <- if (is.null(old_CFG$seed)) 20261026L else old_CFG$seed\n    expected_seed <- as.integer((as.double(seed0) + 104729 * row$NAME_ID) %% 2147483646 + 1)\n    .r4_need(nrow(row) == 1L && identical(request$payload$name, name) &&\n      as.integer(request$payload$NAME_ID) == row$NAME_ID && request$payload$num == 500L &&\n      request$payload$ncores == 10L && identical(request$payload$models, .r4_models) &&\n      request$payload$spin_up_days == row$spin_up_days && request$payload$seed == expected_seed &&\n      identical(calibration$signature, request$signature) && isTRUE(calibration$passed) &&\n      identical(evaluation$signature, request$signature) && isTRUE(evaluation$passed) &&\n      identical(report$signature, request$signature), "Completed two-lake receipt identity mismatch: ", name)\n    ctx <- .r4_cal_integrity(run)\n    .r4_need(identical(ctx$r4_signature, request$signature) && identical(ctx$models, .r4_models),\n      "Completed two-lake context mismatch: ", name)\n    for (model in .r4_models) {\n      .rr_table_equal(ctx$specs[[model]], .r4_expected_specs(model),\n        paste("Completed two-lake parameter design", name, model))\n      expected_pool <- .r4_private(".lw_lhc")(ctx$specs[[model]], 500L,\n        .r4_private(".lw_seed")(expected_seed, model, "LHC"))\n      .rr_table_equal(ctx$pools[[model]], expected_pool, paste("Unchanged 500-set pool", name, model), 0)\n    }\n    cfg <- .r4_private(".lw_read_config")(file.path(config_root, paste0(name, "_LakeEnsemblR.yaml")))\n    .r4_need(identical(.r67_semantic_config(ctx$base, ctx$snapshot),\n      .r67_semantic_config(cfg, config_root)), "Completed lake and new 67-lake scheme inputs/physics differ: ", name)\n    windows <- .r4_private(".lw_windows")(file.path(old_CFG$windows_dir, paste0(name, "_MODIS_windows_reviewed.csv")))\n    .rr_table_equal(ctx$windows, windows, paste("Unchanged reviewed windows", name))\n    contracts <- .r4_private(".lw_contracts")(file.path(old_CFG$contracts_dir,\n      paste0(name, "_output_contracts.csv")), .r4_models, require_confirmed = TRUE)\n    .rr_table_equal(ctx$contracts, contracts, paste("Unchanged time contracts", name))\n    final <- .r4_verify_final(run)\n    .r4_need(identical(evaluation$final_files, final$files) && identical(evaluation$metrics_sha256,\n      .r4_sha(file.path(run, "final_metrics.csv"))), "Completed two-lake final outputs changed: ", name)\n    .r4_private(".lw_check_manifest")(file.path(root, "report"), report$files, "Completed two-lake report")\n    rows[[name]] <- data.frame(name = name, NAME_ID = row$NAME_ID, ncores = 10L, num = 500L,\n      run_dir = run, signature = request$signature, calibration_status = "verified",\n      evaluation_status = "verified", metrics_sha256 = evaluation$metrics_sha256,\n      stringsAsFactors = FALSE)\n    cat("COMPLETED_LAKE_VERIFIED_WITHOUT_RERUN:", name, "\\n")\n  }\n  do.call(rbind, rows)\n}\n\n.r67_atomic_index <- function(index, path) {\n  tmp <- paste0(path, ".new")\n  if (file.exists(tmp)) unlink(tmp)\n  .r4_private(".lw_write_csv")(index, tmp)\n  if (file.exists(path)) .r4_need(unlink(path) == 0L, "Cannot replace progress index; close spreadsheet using it")\n  .r4_need(file.rename(tmp, path), "Cannot install remaining-lake progress index")\n  invisible(index)\n}\n\n.r67_runtime_resume_check <- function(prior, current) {\n  for (field in c("schema", "models", "num", "ncores", "sources", "runtime", "runtime_functions", "helper_functions"))\n    .r4_need(identical(prior$payload[[field]], current$payload[[field]]),\n      "FATAL_SHARED_RUNTIME_OR_SCHEME_CHANGE: ", field,\n      "; preserve this batch and use a new reviewed release/results directory")\n  invisible(TRUE)\n}\n\n.r67_current_runtime_check <- function(live, config_root, ncores, num) {\n  .r67_shared_check(config_root, ncores, num)\n  .r4_check_files(live$payload$sources)\n  runtime <- .r4_private(".lw_package_state")(c("LakeEnsemblR", "FLakeR", "GLM3r", "GOTMr", "SimstratR", "digest", "yaml"))\n  code <- vapply(names(live$payload$runtime_functions), function(nm)\n    .r4_private(".lw_function_hash")(.r4_private(nm)), character(1))\n  helpers <- vapply(names(live$payload$helper_functions), function(nm)\n    .r4_private(".lw_function_hash")(get(nm, mode = "function", inherits = TRUE)), character(1))\n  .r4_need(identical(runtime, live$payload$runtime) && identical(code, live$payload$runtime_functions) &&\n    identical(helpers, live$payload$helper_functions),\n    "FATAL_SHARED_RUNTIME_OR_SCHEME_CHANGE: package/functions changed during the batch")\n  invisible(TRUE)\n}\n\nrun_remaining_lakes_r4 <- function(old_CFG, config_root = .r67_default_config,\n  results_root = "D:/LER4_ice_20261103/n500_c12",\n  completed_root = "D:/LER4_ice_20261102/n500", num = 500L, ncores = 12L,\n  continue_on_error = TRUE) {\n  config_root <- .r4_norm(config_root)\n  registry <- .r67_shared_check(config_root, ncores, num)\n  dir.create(results_root, recursive = TRUE, showWarnings = FALSE)\n  results_root <- .r4_norm(results_root)\n  .r4_need(nchar(results_root, type = "chars") <= 75L && !grepl("[[:space:]]", results_root) &&\n    !identical(results_root, .r4_norm(completed_root)), "Use a short, distinct remaining-67 results directory")\n  if (!is.null(old_CFG$results_root)) .r4_need(!identical(results_root, .r4_norm(old_CFG$results_root)),\n    "Preserve the previous five-model results")\n  .r4_need(is.logical(continue_on_error) && length(continue_on_error) == 1L && !is.na(continue_on_error),\n    "continue_on_error must be TRUE or FALSE")\n  completed <- .r67_verify_completed_two(completed_root, config_root, old_CFG, registry$all)\n  .r4_csv(completed, file.path(results_root, "completed_two_verified.csv"))\n  rows <- registry$remaining\n  .r4_csv(rows, file.path(results_root, "remaining_lakes_effective.csv"))\n  .r4_need(all(nchar(file.path(results_root, rows$name, "v", "p", "_work", "j000007",\n    paste0(rows$name, "_LakeEnsemblR.yaml")), type = "chars") < 240L),\n    "Shorten results_root: a native case path would approach Windows path limits")\n  index <- data.frame(name = rows$name, NAME_ID = rows$NAME_ID,\n    run_dir = file.path(results_root, rows$name, "c"), ncores = 12L, num = 500L,\n    input_status = "not_run", preflight_status = "not_run", calibration_status = "not_run",\n    evaluation_status = "not_run", report_status = "not_run", error = "", stringsAsFactors = FALSE)\n  index_path <- file.path(results_root, "remaining_lake_run_index.csv")\n  if (file.exists(index_path)) {\n    before <- .rr_read_csv(index_path)\n    .r4_need(identical(as.character(before$name), rows$name) &&\n      identical(as.integer(before$NAME_ID), rows$NAME_ID), "Existing remaining-batch index registry changed")\n    for (field in intersect(names(index), names(before))) index[[field]] <- before[[field]]\n  }\n  write_index <- function() .r67_atomic_index(index, index_path)\n  write_index()\n  lives <- vector("list", nrow(rows))\n  # No engine starts until all 67 input/identity checks and shared-state checks finish.\n  for (i in seq_len(nrow(rows))) {\n    cat("Input audit", i, "/67:", rows$name[i], "\\n")\n    live <- tryCatch(.r4_live(old_CFG, config_root, rows[i, , drop = FALSE], num, ncores), error = identity)\n    if (inherits(live, "error")) {\n      index$input_status[i] <- "failed"; index$error[i] <- conditionMessage(live); write_index()\n      if (!continue_on_error) stop(live)\n      next\n    }\n    request <- file.path(results_root, rows$name[i], "request.rds")\n    if (file.exists(request)) .r67_runtime_resume_check(.r4_read(request), live)\n    lives[[i]] <- live; index$input_status[i] <- "ok"; index$error[i] <- ""; write_index()\n  }\n  r4_prepare_local_parallel(as.integer(ncores))\n  for (i in seq_len(nrow(rows))) {\n    if (is.null(lives[[i]])) next\n    live <- lives[[i]]; root <- file.path(results_root, rows$name[i]); run <- file.path(root, "c")\n    .r67_current_runtime_check(live, config_root, ncores, num)\n    dir.create(root, recursive = TRUE, showWarnings = FALSE)\n    request <- file.path(root, "request.rds")\n    cat("REMAINING_LAKE", i, "/67:", rows$name[i], "\\n")\n    active <- "input_status"\n    tryCatch({\n      fresh <- .r4_live(old_CFG, config_root, rows[i, , drop = FALSE], num, ncores)\n      .r4_need(identical(fresh$payload, live$payload),\n        "Lake input changed after its startup audit; preserve its results and review: ", rows$name[i])\n      active <- "initialization"\n      if (file.exists(request)) {\n        prior <- .r4_read(request)\n        .r4_need(identical(prior$signature, live$signature) && identical(prior$payload, live$payload),\n          "Saved lake input/design signature differs; preserve its results and review the changed input")\n        ctx <- .r4_load_context(run, root, live$signature)\n        .r4_need(identical(ctx$r4_signature, live$signature) && identical(ctx$models, .r4_models),\n          "Remaining-lake context request mismatch")\n        .r4_private(".lw_verify_runtime")(ctx)\n      } else {\n        if (dir.exists(run)) .r4_archive(run, .r4_evidence(file.path(root, "e"), "init"), "interrupted_unsealed_initialization")\n        ctx <- .r4_new_context(live, run)\n        .r4_save(list(schema = .r4_schema, signature = live$signature, payload = live$payload), request)\n      }\n      active <- "preflight_status"\n      .r4_phase(file.path(root, "phase_times.csv"), "preflight_and_changed_model_replay", {\n        .r4_preflight(ctx, root, live$signature, ncores)\n      })\n      index$preflight_status[i] <- "ok"; write_index()\n      active <- "calibration_status"\n      ctx <- .r4_calibrate(ctx, root, live, ncores)\n      index$calibration_status[i] <- "ok"; write_index()\n      active <- "evaluation_status"\n      .r4_evaluate(ctx, root, live$signature)\n      index$evaluation_status[i] <- "ok"; write_index()\n      active <- "report_status"\n      .r4_report(ctx, root)\n      index$report_status[i] <- "ok"; index$error[i] <- ""; write_index()\n    }, error = function(e) {\n      if (active %in% names(index)) index[[active]][i] <<- "failed"\n      index$error[i] <<- conditionMessage(e); write_index()\n      cat("LAKE_FAILED_AND_RECORDED:", rows$name[i], "\\n", conditionMessage(e), "\\n")\n      if (!continue_on_error) stop(e)\n    })\n  }\n  passed <- index$input_status == "ok" & index$preflight_status == "ok" &\n    index$calibration_status == "ok" & index$evaluation_status == "ok" & index$report_status == "ok" & index$error == ""\n  .r67_atomic_index(index[!passed, , drop = FALSE], file.path(results_root, "remaining_failed_lakes.csv"))\n  if (!all(passed)) {\n    stop("REMAINING_67_NOT_ALL_PASSED; completed ", sum(passed), "/67; inspect remaining_failed_lakes.csv. Completed results are retained.", call. = FALSE)\n  }\n  combined <- rbind(data.frame(name = completed$name, NAME_ID = completed$NAME_ID, run_dir = completed$run_dir,\n    ncores = 10L, num = 500L, calibration_status = "ok", evaluation_status = "ok", source_batch = "completed_two"),\n    data.frame(name = index$name, NAME_ID = index$NAME_ID, run_dir = index$run_dir, ncores = 12L, num = 500L,\n      calibration_status = "ok", evaluation_status = "ok", source_batch = "remaining67"))\n  combined <- combined[order(combined$name), , drop = FALSE]; rownames(combined) <- NULL\n  .r4_need(nrow(combined) == 69L && !anyDuplicated(combined$name), "Combined 69-lake index is incomplete")\n  .r4_csv(combined, file.path(results_root, "combined_69_lake_run_index.csv"))\n  cat("REMAINING_67_FOUR_MODELS_CALIBRATION_EVALUATION_PASSED\\n")\n  cat("ALL_69_FOUR_MODELS_CALIBRATION_EVALUATION_PASSED\\nResults:", results_root, "\\n")\n  invisible(index)\n}\n', 'Analyze_two_lakes_R4.R': '# Read-only analysis of completed four-model runs. No engines or model inputs are changed.\n.a4_need <- function(ok, ...) if (!isTRUE(ok)) stop(..., call. = FALSE)\n.a4_read <- function(p) {\n  .a4_need(file.exists(p), "Missing analysis input: ", p)\n  utils::read.csv(p, stringsAsFactors = FALSE, check.names = FALSE, fileEncoding = "UTF-8-BOM")\n}\n.a4_sha <- function(p) digest::digest(file = p, algo = "sha256", serialize = FALSE)\n.a4_csv <- function(x, p) utils::write.csv(x, p, row.names = FALSE, fileEncoding = "UTF-8", na = "NA")\n.a4_receipt <- function(p) {\n  .a4_need(file.exists(p) && file.exists(paste0(p, ".sha256")), "Missing receipt/seal: ", p)\n  .a4_need(identical(readLines(paste0(p, ".sha256"), warn = FALSE), .a4_sha(p)), "Receipt hash differs: ", p)\n  readRDS(p)\n}\n.a4_report_check <- function(lake_dir) {\n  record <- .a4_receipt(file.path(lake_dir, "report.rds"))\n  root <- file.path(lake_dir, "report")\n  expected <- record$files\n  current <- sort(list.files(root, recursive = TRUE, all.files = TRUE, no.. = TRUE))\n  .a4_need(!anyDuplicated(expected$file) && setequal(current, as.character(expected$file)), "Report file list changed: ", root)\n  for (i in seq_len(nrow(expected))) {\n    p <- file.path(root, expected$file[i])\n    .a4_need(file.info(p)$size == expected$bytes[i] && identical(.a4_sha(p), expected$sha256[i]),\n      "Report artifact changed: ", p)\n  }\n  invisible(record)\n}\n.a4_metrics <- function(x) {\n  e <- x$predicted - x$observed\n  data.frame(N = nrow(x), RMSE = sqrt(mean(e^2)), MAE = mean(abs(e)), bias = mean(e),\n    NSE = if (sum((x$observed - mean(x$observed))^2) > 0) 1 - sum(e^2)/sum((x$observed - mean(x$observed))^2) else NA_real_,\n    r = if (nrow(x) >= 2L && stats::sd(x$observed) > 0 && stats::sd(x$predicted) > 0) stats::cor(x$observed, x$predicted) else NA_real_)\n}\n.a4_figure <- function(out, name, draw, width = 1800, height = 1100) {\n  for (ext in c("png", "pdf")) {\n    p <- file.path(out, paste0(name, ".", ext))\n    if (ext == "png") grDevices::png(p, width = width, height = height, res = 160)\n    else grDevices::pdf(p, width = width/160, height = height/160, useDingbats = FALSE)\n    tryCatch(draw(), finally = grDevices::dev.off())\n  }\n}\n\nanalyze_two_lakes_r4 <- function(run_root = "D:/LER4_ice_20261102/n500", out_dir = NULL, old_akesaiqin_metrics = NULL) {\n  .a4_need(requireNamespace("digest", quietly = TRUE), "Load the original isolated R library containing digest first")\n  run_root <- normalizePath(run_root, winslash = "/", mustWork = TRUE)\n  lakes <- c("akesaiqin", "namco")\n  models <- c("FLake", "GLM", "GOTM", "Simstrat", "Ensemble4")\n  colors <- setNames(c("#3686A0", "#E3A437", "#4D9D70", "#A06FAB", "#444444"), models)\n  index <- .a4_read(file.path(run_root, "two_lake_run_index.csv"))\n  index <- index[match(lakes, index$name), , drop = FALSE]\n  .a4_need(identical(as.character(index$name), lakes) && all(index$calibration_status == "ok") &&\n    all(index$evaluation_status == "ok"), "Both lakes must have completed calibration and evaluation")\n  if (is.null(out_dir)) out_dir <- file.path(dirname(run_root), paste0("analysis_two_lakes_", format(Sys.time(), "%Y%m%dT%H%M%S")))\n  .a4_need(!file.exists(out_dir), "Use a NEW analysis output directory: ", out_dir)\n  .a4_need(dir.create(out_dir, recursive = TRUE), "Cannot create analysis output directory")\n  out_dir <- normalizePath(out_dir, winslash = "/", mustWork = TRUE)\n  metric_list <- pair_list <- param_list <- response_list <- timing_list <- list()\n  for (lake in lakes) {\n    root <- file.path(run_root, lake); report <- file.path(root, "report")\n    .a4_report_check(root)\n    metrics <- .a4_read(file.path(report, "model_metrics.csv"))\n    ensemble <- .a4_read(file.path(report, "ensemble4_metrics.csv"))\n    fields <- c("model", "split", "N", "RMSE", "MAE", "bias", "NSE", "r")\n    metrics <- rbind(metrics[fields], ensemble[fields])\n    .a4_need(nrow(metrics) == 10L && setequal(metrics$model, models), "Unexpected metrics/model design: ", lake)\n    pairs <- .a4_read(file.path(report, "pairs_models_ensemble4.csv"))\n    .a4_need(setequal(pairs$model, models) && all(is.finite(pairs$predicted)) && all(is.finite(pairs$observed)),\n      "Expected finite four-model plus ensemble window pairs: ", lake)\n    .a4_need(!anyDuplicated(paste(pairs$model, pairs$split, pairs$window_id)), "Duplicate model/split/window: ", lake)\n    pairs$start_UTC <- as.POSIXct(pairs$window_start, tz = "UTC")\n    pairs$end_UTC <- as.POSIXct(pairs$window_end, tz = "UTC")\n    .a4_need(all(!is.na(pairs$start_UTC) & !is.na(pairs$end_UTC) & pairs$end_UTC > pairs$start_UTC), "Invalid window dates")\n    pairs$midpoint_UTC <- pairs$start_UTC + as.numeric(difftime(pairs$end_UTC, pairs$start_UTC, units = "secs"))/2\n    pairs$month <- as.integer(format(pairs$midpoint_UTC, "%m", tz = "UTC"))\n    pairs$residual <- pairs$predicted - pairs$observed\n    for (i in seq_len(nrow(metrics))) {\n      z <- pairs[pairs$model == metrics$model[i] & pairs$split == metrics$split[i], , drop = FALSE]\n      calc <- .a4_metrics(z)\n      .a4_need(isTRUE(all.equal(as.numeric(unlist(metrics[i, names(calc)])), as.numeric(unlist(calc[1, ])), tolerance = 1e-9)),\n        "Reported metrics do not reproduce from equal-weight window pairs: ", lake, "/", metrics$model[i], "/", metrics$split[i])\n    }\n    for (split in c("calibration", "validation")) {\n      ref <- pairs[pairs$model == models[1] & pairs$split == split, , drop = FALSE]\n      for (model in models[-1]) {\n        z <- pairs[pairs$model == model & pairs$split == split, , drop = FALSE]\n        .a4_need(setequal(z$window_id, ref$window_id) && isTRUE(all.equal(z$observed[match(ref$window_id, z$window_id)], ref$observed)),\n          "Models do not share identical observed windows: ", lake)\n      }\n    }\n    parameters <- .a4_read(file.path(report, "selected_parameters.csv"))\n    .a4_need(nrow(parameters) == 9L && sum(parameters$model == "Simstrat") == 3L, "Expected 2+2+2+3 selected parameters")\n    parameters$lower <- ifelse(parameters$parameter == "ModelParameters/p_sw_ice", 0.9, 0.5)\n    parameters$upper <- ifelse(parameters$parameter == "wind_speed", 2, ifelse(parameters$parameter == "swr", 1.5, 3))\n    parameters$fraction_of_range <- (parameters$value - parameters$lower)/(parameters$upper - parameters$lower)\n    parameters$within_5pct_of_boundary <- parameters$fraction_of_range <= 0.05 | parameters$fraction_of_range >= 0.95\n    response <- .a4_read(file.path(report, "parameter_response.csv"))\n    metrics$lake <- pairs$lake <- parameters$lake <- response$lake <- lake\n    metric_list[[lake]] <- metrics; pair_list[[lake]] <- pairs\n    param_list[[lake]] <- parameters; response_list[[lake]] <- response\n    timing <- .a4_read(file.path(root, "phase_times.csv")); timing$lake <- lake; timing_list[[lake]] <- timing\n  }\n  metrics <- do.call(rbind, metric_list); pairs <- do.call(rbind, pair_list)\n  parameters <- do.call(rbind, param_list); response <- do.call(rbind, response_list)\n  timing <- do.call(rbind, timing_list)\n  .a4_csv(metrics, file.path(out_dir, "metrics_two_lakes.csv"))\n  .a4_csv(pairs, file.path(out_dir, "matched_window_pairs_two_lakes.csv"))\n  .a4_csv(parameters, file.path(out_dir, "selected_parameters_and_boundaries.csv"))\n  .a4_csv(response, file.path(out_dir, "calibration_parameter_responses.csv"))\n  .a4_csv(timing, file.path(out_dir, "phase_times_all_attempts.csv"))\n  elapsed <- stats::aggregate(timing$elapsed_seconds, timing[c("lake", "phase", "status")], sum)\n  names(elapsed)[ncol(elapsed)] <- "elapsed_seconds"\n  .a4_csv(elapsed, file.path(out_dir, "phase_elapsed_summary.csv"))\n  spans <- do.call(rbind, lapply(lakes, function(lake) {\n    z <- timing[timing$lake == lake, ]; a <- as.POSIXct(z$start_UTC, tz = "UTC"); b <- as.POSIXct(z$end_UTC, tz = "UTC")\n    data.frame(lake = lake, first_phase_start_UTC = format(min(a), "%Y-%m-%d %H:%M:%S", tz = "UTC"),\n      last_phase_end_UTC = format(max(b), "%Y-%m-%d %H:%M:%S", tz = "UTC"),\n      phase_span_seconds = as.numeric(difftime(max(b), min(a), units = "secs")),\n      sum_phase_elapsed_seconds = sum(z$elapsed_seconds), phase_records = nrow(z))\n  }))\n  .a4_csv(spans, file.path(out_dir, "phase_wall_span_summary.csv"))\n  groups <- split(pairs[pairs$split == "validation", ], interaction(pairs$lake[pairs$split == "validation"],\n    pairs$model[pairs$split == "validation"], pairs$month[pairs$split == "validation"], drop = TRUE))\n  monthly <- do.call(rbind, lapply(groups, function(z) cbind(data.frame(lake = z$lake[1], model = z$model[1],\n    window_midpoint_month = z$month[1]), .a4_metrics(z))))\n  .a4_csv(monthly, file.path(out_dir, "monthly_validation_errors.csv"))\n  sim <- response[response$model == "Simstrat", ]\n  triplets <- sim[sim$parameter == "wind_speed", c("lake", "par_id", "status", "RMSE", "value")]\n  names(triplets)[names(triplets) == "value"] <- "wind_speed"\n  for (parameter in c("swr", "ModelParameters/p_sw_ice")) {\n    z <- sim[sim$parameter == parameter, c("lake", "par_id", "value")]\n    names(z)[3] <- if (parameter == "swr") "swr" else "p_sw_ice"\n    triplets <- merge(triplets, z, by = c("lake", "par_id"), all = TRUE)\n  }\n  .a4_need(all(is.finite(triplets$wind_speed) & is.finite(triplets$swr) & is.finite(triplets$p_sw_ice)),\n    "Incomplete Simstrat parameter triplets")\n  .a4_csv(triplets, file.path(out_dir, "simstrat_parameter_triplets.csv"))\n  if (!is.null(old_akesaiqin_metrics)) {\n    old <- .a4_read(old_akesaiqin_metrics)\n    old <- old[old$model %in% models[1:4], fields]\n    comparison <- merge(old, metrics[metrics$lake == "akesaiqin", fields], by = c("model", "split"),\n      suffixes = c("_old_five_model_scheme", "_new_four_model_ice_scheme"))\n    comparison$RMSE_new_minus_old <- comparison$RMSE_new_four_model_ice_scheme - comparison$RMSE_old_five_model_scheme\n    .a4_csv(comparison, file.path(out_dir, "akesaiqin_old_vs_new_individual_models.csv"))\n  }\n  val <- metrics[metrics$split == "validation", ]; val <- val[order(val$lake, val$RMSE), ]\n  val$descriptive_RMSE_rank <- ave(val$RMSE, val$lake, FUN = function(x) rank(x, ties.method = "min"))\n  .a4_csv(val, file.path(out_dir, "validation_comparison.csv"))\n  .a4_figure(out_dir, "01_RMSE_by_lake_and_split", function() {\n    graphics::par(mfrow = c(1, 2), mar = c(7, 5, 3, 1))\n    for (lake in lakes) {\n      z <- metric_list[[lake]]\n      # Explicit indexing avoids assuming report row order.\n      a <- sapply(models, function(m) vapply(c("calibration", "validation"), function(s) z$RMSE[z$model == m & z$split == s], numeric(1)))\n      graphics::barplot(a, beside = TRUE, names.arg = models, las = 2, col = c("#547E9D", "#CF8B4A"),\n        ylab = "Equal-window RMSE (deg C)", main = lake)\n      graphics::legend("topleft", c("Calibration", "Validation"), fill = c("#547E9D", "#CF8B4A"), bty = "n")\n    }\n  })\n  .a4_figure(out_dir, "02_validation_observed_predicted", function() {\n    graphics::par(mfrow = c(2, 5), mar = c(4, 4, 3, 1))\n    for (lake in lakes) for (model in models) {\n      z <- pairs[pairs$lake == lake & pairs$model == model & pairs$split == "validation", ]\n      limits <- range(c(z$observed, z$predicted))\n      graphics::plot(z$observed, z$predicted, xlim = limits, ylim = limits, asp = 1, pch = 16, cex = 0.5,\n        col = grDevices::adjustcolor(colors[model], 0.55), xlab = "Observed window mean (deg C)", ylab = "Predicted window mean (deg C)", main = paste(lake, model))\n      graphics::abline(0, 1, col = "grey40", lty = 2)\n    }\n  }, width = 2300, height = 1050)\n  .a4_figure(out_dir, "03_validation_window_temperature", function() {\n    graphics::par(mfrow = c(2, 1), mar = c(4, 5, 3, 1))\n    for (lake in lakes) {\n      z <- pairs[pairs$lake == lake & pairs$split == "validation", ]\n      graphics::plot(as.Date(z$midpoint_UTC), z$predicted, type = "n", ylim = range(c(z$observed, z$predicted)),\n        xlab = "Window midpoint (UTC)", ylab = "Window mean temperature (deg C)", main = lake)\n      for (model in models) {\n        x <- z[z$model == model, ]; graphics::points(as.Date(x$midpoint_UTC), x$predicted, col = colors[model], pch = 16, cex = 0.45)\n      }\n      x <- z[z$model == models[1], ]; graphics::points(as.Date(x$midpoint_UTC), x$observed, pch = 4, cex = 0.6)\n      graphics::legend("topleft", c("Observed", models), col = c("black", colors), pch = c(4, rep(16, 5)), ncol = 3, cex = 0.7, bty = "n")\n    }\n  })\n  .a4_figure(out_dir, "04_validation_residual_by_window_month", function() {\n    graphics::par(mfrow = c(2, 5), mar = c(4, 4, 3, 1))\n    for (lake in lakes) for (model in models) {\n      z <- pairs[pairs$lake == lake & pairs$model == model & pairs$split == "validation", ]\n      groups <- lapply(1:12, function(m) z$residual[z$month == m])\n      graphics::boxplot(groups, names = 1:12, outline = FALSE, col = colors[model], xlab = "Window midpoint month",\n        ylab = "Predicted - observed (deg C)", main = paste(lake, model))\n      graphics::abline(h = 0, lty = 2, col = "grey40")\n    }\n  }, width = 2300, height = 1050)\n  for (lake in lakes) .a4_figure(out_dir, paste0("05_parameter_responses_", lake), function() {\n    graphics::par(mfrow = c(3, 3), mar = c(4, 4, 3, 1))\n    pars <- param_list[[lake]]\n    for (i in seq_len(nrow(pars))) {\n      z <- response[response$lake == lake & response$model == pars$model[i] & response$parameter == pars$parameter[i] &\n        response$status == "ok" & is.finite(response$RMSE), ]\n      graphics::plot(z$value, z$RMSE, pch = 16, cex = 0.35, col = grDevices::adjustcolor(colors[pars$model[i]], 0.45),\n        xlab = pars$parameter[i], ylab = "Calibration RMSE (deg C)", main = paste(lake, pars$model[i]))\n      graphics::abline(v = pars$value[i], col = "#B51C29", lty = 2)\n    }\n  }, width = 1800, height = 1500)\n  .a4_figure(out_dir, "06_simstrat_joint_parameter_responses", function() {\n    graphics::par(mfrow = c(2, 2), mar = c(4, 4, 3, 1))\n    palette <- c("#254D85", "#3389A5", "#72B9A0", "#DFBD62", "#BD624B")\n    for (lake in lakes) for (parameter in c("wind_speed", "swr")) {\n      z <- triplets[triplets$lake == lake & triplets$status == "ok" & is.finite(triplets$RMSE), ]\n      bin <- pmax(1L, pmin(5L, ceiling(5 * rank(z$RMSE, ties.method = "average")/nrow(z))))\n      graphics::plot(z[[parameter]], z$p_sw_ice, col = grDevices::adjustcolor(palette[bin], 0.65), pch = 16, cex = 0.55,\n        xlab = parameter, ylab = "p_sw_ice", main = paste(lake, "Simstrat"))\n      selected <- param_list[[lake]]; selected <- selected[selected$model == "Simstrat", ]\n      graphics::points(selected$value[selected$parameter == parameter], selected$value[selected$parameter == "ModelParameters/p_sw_ice"], pch = 8, cex = 1.5, lwd = 2)\n      graphics::legend("topleft", legend = c("Lowest 20% RMSE", "20-40%", "40-60%", "60-80%", "Highest 20% RMSE"),\n        col = palette, pch = 16, bty = "n", cex = 0.6)\n    }\n  })\n  note <- c("Two-lake four-model analysis. Report receipt and report-file SHA256 manifests verified.",\n    "Metrics independently reproduced from equal-weight MODIS window pairs. Validation ranking is descriptive; parameter selection uses calibration only.",\n    "Points represent matched window means, not daily observations. No continuous winter series is inferred across missing windows.",\n    "Monthly residuals use window midpoint month. Temperature-proxy ice-free labels are not independent ice observations.",\n    "p_sw_ice=0.9-3 is the requested exploratory range, not an official physical recommendation; simultaneous swr calibration may compensate.",\n    "Parameter-response scatter is not Feldbauer delta/S1 sensitivity analysis. No new engine runs or bootstrap are performed.",\n    "All original run artifacts are read-only. Existing full candidate/native integrity is not rehashed by this plotting script.")\n  writeLines(note, file.path(out_dir, "READ_ME.txt"))\n  cat("TWO_LAKE_ANALYSIS_AND_PLOTS_COMPLETE\\n", out_dir, "\\n", sep = "")\n  invisible(out_dir)\n}\n', 'Collect_two_lake_results.py': '"""Collect small completed-run evidence without running R or editing any result."""\nfrom pathlib import Path\nimport csv\nimport datetime as dt\nimport hashlib\nimport io\nimport json\nimport os\nimport zipfile\n\ndef local_path(value):\n    text = str(value).replace("\\\\", "/")\n    if os.name != "nt" and len(text) >= 3 and text[1:3] == ":/":\n        text = "/mnt/" + text[0].lower() + text[2:]\n    return Path(text)\n\ndef collect_two_lake_results(\n    run_root="D:/LER4_ice_20261102/n500",\n    config_root="C:/Geography/R/dissertation_LakeEnsemblR/yaml/r4_ice_20261102",\n    include_ice=True,\n):\n    root, config = local_path(run_root), local_path(config_root)\n    index_file = root / "two_lake_run_index.csv"\n    with index_file.open(encoding="utf-8-sig", newline="") as handle:\n        rows = list(csv.DictReader(handle))\n    selected = {row["name"]: row for row in rows}\n    lakes = ("akesaiqin", "namco")\n    if any(name not in selected or selected[name]["calibration_status"] != "ok" or\n           selected[name]["evaluation_status"] != "ok" for name in lakes):\n        raise ValueError("两湖率定与评价尚未全部完成；未打包。")\n    files, skipped = {}, []\n\n    def add(path, archive_name, required=False, max_mb=8):\n        if not path.is_file():\n            if required:\n                raise FileNotFoundError(path)\n            return\n        if path.stat().st_size > max_mb * 1024**2:\n            if required:\n                raise ValueError(f"必要文件超过{max_mb}MB，请保留原文件并核对: {path}")\n            skipped.append({"file": str(path), "reason": f"larger than {max_mb}MiB"})\n            return\n        if path.is_symlink():\n            raise ValueError(f"不接受符号链接: {path}")\n        files[str(archive_name).replace("\\\\", "/")] = path\n\n    add(index_file, index_file.name, required=True)\n    report_names = ("model_metrics.csv", "ensemble4_metrics.csv", "selected_parameters.csv",\n                    "parameter_response.csv", "pairs_models_ensemble4.csv", "RMSE_by_split.png", "READ_ME.txt")\n    for lake in lakes:\n        base = root / lake\n        for filename in report_names:\n            add(base / "report" / filename, f"{lake}/report/{filename}", required=True)\n        add(base / "phase_times.csv", f"{lake}/phase_times.csv", required=True)\n        for receipt in ("request.rds", "preflight.rds", "calibration.rds", "evaluation.rds", "report.rds"):\n            for ending in ("", ".sha256"):\n                name = receipt + ending\n                add(base / name, f"{lake}/{name}", required=True)\n        for name in ("serial_status.csv", "parallel_status.csv", "workers.log"):\n            add(base / "v" / name, f"{lake}/v/{name}")\n        small = ("final_metrics.csv", "best_by_calibration_RMSE.csv", "candidate_success_fractions.csv",\n                 "calibration_summary.csv", "context.rds", "context_integrity.csv", "metadata.rds",\n                 "engine_packages.csv", "function_manifest.csv", "sessionInfo.txt", "workflow_sources.csv",\n                 "live_input_files.csv", "workflow_R4ice.R", "recovery_helper_rev3.R")\n        for name in small:\n            add(base / "c" / name, f"{lake}/c/{name}")\n        for model in ("FLake", "GLM", "GOTM", "Simstrat"):\n            for prefix in ("params_", "spec_"):\n                name = prefix + model + ".csv"\n                add(base / "c" / name, f"{lake}/c/{name}", required=True)\n        if include_ice:\n            sim = base / "c" / "final" / "Simstrat"\n            for path in sorted(sim.rglob("*")):\n                if not path.is_file():\n                    continue\n                name = path.name.lower()\n                ice = path.suffix.lower() == ".dat" and ("iceh" in name or "snowh" in name)\n                metadata = name in {"simstrat.par", "native_config.par", "native_surface.rds", "status.csv",\n                                    "contract.csv", "engine_console.log", "warnings.txt"}\n                if ice or metadata:\n                    add(path, path.relative_to(root).as_posix(), max_mb=5)\n    for name in ("selected_lakes.csv", "selected_parameter_design.csv", "_yaml_settings.json",\n                 "_puhu_geometry_audit.json", "_runner_sha256.json"):\n        add(config / name, "configuration/" + name)\n    for lake in lakes:\n        name = lake + "_LakeEnsemblR.yaml"\n        add(config / name, "configuration/" + name)\n    # Include the most recent optional derived analysis, if already created.\n    analyses = sorted(p for p in root.parent.glob("analysis_two_lakes_*") if p.is_dir())\n    if analyses:\n        for path in sorted(analyses[-1].glob("*")):\n            if path.is_file():\n                add(path, "derived_analysis/" + path.name)\n    target_dir = root.parent / "deliveries"\n    target_dir.mkdir(parents=True, exist_ok=True)\n    stamp = dt.datetime.now().strftime("%Y%m%dT%H%M%S")\n    target = target_dir / f"two_lake_results_{stamp}.zip"\n    manifest = []\n    with zipfile.ZipFile(target, "x", compression=zipfile.ZIP_DEFLATED) as archive:\n        for name, path in sorted(files.items()):\n            data = path.read_bytes()\n            archive.writestr(name, data)\n            manifest.append({"file": name, "bytes": len(data), "sha256": hashlib.sha256(data).hexdigest()})\n        text = io.StringIO(newline="")\n        writer = csv.DictWriter(text, fieldnames=("file", "bytes", "sha256"), lineterminator="\\n")\n        writer.writeheader(); writer.writerows(manifest)\n        archive.writestr("COLLECTION_SHA256.csv", text.getvalue().encode("utf-8-sig"))\n        archive.writestr("COLLECTION_NOTES.json", json.dumps({"run_root": str(root), "lakes": lakes,\n            "skipped": skipped, "scope": "read-only small result evidence; no forcing, executable, or full candidate folders"},\n            ensure_ascii=False, indent=2).encode("utf-8"))\n    print("TWO_LAKE_RESULTS_ZIP_READY")\n    print(target)\n    print(f"{target.stat().st_size / 1024**2:.2f} MB; {len(files)} files")\n    return target\n\nif __name__ == "__main__":\n    collect_two_lake_results()\n'}
entry = f"""# Starts/resumes only the remaining67. No packages are installed or replaced.
ISO <- {json.dumps(windows_text(host_path(ISO)), ensure_ascii=False)}
source(file.path(ISO, "Restart_LER_window.R"), encoding = "UTF-8")
CONFIG4 <- {json.dumps(windows_text(config_root), ensure_ascii=False)}
HELPER4 <- normalizePath(file.path(CONFIG4, "Resume_interrupted_calibration_9003_rev3.R"), winslash = "/", mustWork = TRUE)
RUNNER4 <- normalizePath(file.path(CONFIG4, "Run_remaining67_R4ice_12.R"), winslash = "/", mustWork = TRUE)
R4 <- new.env(parent = .GlobalEnv)
source(HELPER4, local = R4, encoding = "UTF-8")
source(RUNNER4, local = R4, encoding = "UTF-8")
R4$.rr_resume_source_file <- HELPER4
R4$.r4_source <- RUNNER4
stopifnot(identical(R4$.r4_sha(HELPER4), "74cfbe6a90da76df42c80b168edb45d4a1e78ba08dceac4976ffff84516e5afa"))
RESULT67 <- R4$run_remaining_lakes_r4(old_CFG = CFG, config_root = CONFIG4,
    results_root = "D:/LER4_ice_20261103/n500_c12",
    completed_root = "D:/LER4_ice_20261102/n500", num = 500L,
    ncores = 12L, continue_on_error = TRUE)
"""
SCRIPT_CONTENTS["START_remaining67_core12.R"] = entry
analysis_entry = f"""# Analysis only; no model engines are run.
ISO <- {json.dumps(windows_text(host_path(ISO)), ensure_ascii=False)}
source(file.path(ISO, "Restart_LER_window.R"), encoding = "UTF-8")
A4 <- new.env(parent = .GlobalEnv)
source({json.dumps(windows_text(config_root / "Analyze_two_lakes_R4.R"), ensure_ascii=False)}, local = A4, encoding = "UTF-8")
ANALYSIS4 <- A4$analyze_two_lakes_r4("D:/LER4_ice_20261102/n500")
"""
SCRIPT_CONTENTS["ANALYZE_completed_two.R"] = analysis_entry
for name, text in SCRIPT_CONTENTS.items():
    path = config_root / name
    require(not path.exists() or path.read_bytes() == text.encode("utf-8"),
            f"已有不同脚本，停止以免覆盖: {path}")
digests = {name: save_identical_or_new(config_root / name, text)
           for name, text in SCRIPT_CONTENTS.items()}
save_identical_or_new(config_root / "_runner_sha256.json",
                     json.dumps(digests, ensure_ascii=False, indent=2) + "\n")
print("REMAINING67_RUNNER_FILES_READY_NO_MODELS_STARTED")
print("正式启动命令：")
print('source(' + json.dumps(windows_text(config_root / "START_remaining67_core12.R")) +
      ', encoding = "UTF-8")')


## 在原隔离项目RStudio正式启动

先Session → Restart R，然后运行：

```r
source("C:/Geography/R/dissertation_LakeEnsemblR/yaml/r4_ice_20261103_core12/START_remaining67_core12.R", encoding = "UTF-8")
```

入口会先做12进程端口/通信测试、只读核验旧两湖与其参数池，并审计其余67湖输入。
随后逐湖执行新参数导出与串并行检查、正式500套率定、最优参数评价及报告。
单湖失败会记录并继续其他湖，共享软件/脚本变化则停止。错误见remaining_failed_lakes.csv。

最终必须看到：

```
REMAINING_67_FOUR_MODELS_CALIBRATION_EVALUATION_PASSED
ALL_69_FOUR_MODELS_CALIBRATION_EVALUATION_PASSED
```

中断后保持输入、包、代码和进程数不变，重新执行同一source入口，只计算缺失候选。
进度文件：D:/LER4_ice_20261103/n500_c12/remaining_lake_run_index.csv。
全部通过后生成combined_69_lake_run_index.csv，指向原两湖及新67湖结果。

两湖分析可在正式启动67湖之前运行ANALYZE_completed_two.R；它只写新分析目录，不运行模型。
也可使用单独的Analyze_and_collect_two_lakes_20261103.ipynb进行分析及小包导出。
